# Temperature upper-slack sensitivity

This notebook reproduces the observed-future MIQP sensitivity study at exactly
`w_x_slack = 100, 1000, 10000`. It uses the same fixed baselines, thermal
surrogates, PV coefficients, and 22-date study split as the main study.


The measured L14 table is private and is not distributed. Supply it at
`data/private/l14_merged_data_with_rain.csv`. Outputs are written to
`outputs/simulations/weight_sweep/`. Gurobi is discovered normally from the
active Python environment.


In [ ]:
import json
import os
import platform
import sys
from collections import OrderedDict
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "mmv4ef").is_dir() and (candidate / "models").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the MMV4EF release repository.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from mmv4ef.mpc_future_data import apply_future_data_source, build_forecast_audit
from mmv4ef.temperature_bound_sweep import (
    DenominatorAwareRatioContract,
    SWEEP_WEIGHTS as _LIBRARY_SWEEP_WEIGHTS,
    build_balanced_selection,
    find_nonfinite_numeric_failures,
    format_validation_failures,
    optimize_miqp_with_numeric_retry,
    temperature_metrics,
    validate_denominator_aware_ratios,
)

SWEEP_WEIGHTS = (100.0, 1000.0, 10000.0)
if tuple(_LIBRARY_SWEEP_WEIGHTS) != SWEEP_WEIGHTS:
    raise RuntimeError("Notebook and library sweep weights do not match.")

L14_DATA_PATH = REPO_ROOT / "data" / "private" / "l14_merged_data_with_rain.csv"
AC_MODEL_PATH = REPO_ROOT / "models" / "thermal" / "ac_model.pth"
NV_MODEL_PATH = REPO_ROOT / "models" / "thermal" / "nv_model.pth"
PV_COEFFICIENT_PATH = REPO_ROOT / "models" / "pv" / "pv_appendix_b_best_model.csv"
OUTPUT_DIR = REPO_ROOT / "outputs" / "simulations" / "weight_sweep"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (12, 4)


In [ ]:
# Surrogate utilities (same architecture as training notebooks)
class CNNLSTM(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        self.conv = nn.Conv1d(in_channels=input_dim, out_channels=hidden_dim, kernel_size=1)
        self.lstm = nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x.transpose(1, 2)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out


def load_surrogate(path: str, device: torch.device):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = CNNLSTM(ckpt['input_dim'], ckpt['hidden_dim'], ckpt['output_dim']).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model, ckpt['scaler_X'], ckpt['scaler_U']


# scaler helpers reused for forward + linearization
def _scaler_params_to_torch(scaler, device: torch.device):
    scale = torch.as_tensor(scaler.scale_, dtype=torch.float32, device=device)
    offset = torch.as_tensor(scaler.min_, dtype=torch.float32, device=device)
    return scale, offset


@torch.inference_mode()
def forward_surrogate(model: CNNLSTM, scaler_X, scaler_U, x, u, device: torch.device):
    x0 = torch.as_tensor(np.array(x, dtype=np.float32, copy=True), dtype=torch.float32, device=device).view(1, -1)
    u0 = torch.as_tensor(np.array(u, dtype=np.float32, copy=True), dtype=torch.float32, device=device).view(1, -1)
    x_scale, x_min = _scaler_params_to_torch(scaler_X, device)
    u_scale, u_min = _scaler_params_to_torch(scaler_U, device)

    x_norm = x0 * x_scale + x_min
    u_norm = u0 * u_scale + u_min
    model_in = torch.cat([x_norm, u_norm], dim=1).unsqueeze(1)
    x_next_norm = model(model_in).squeeze(0)
    x_next = (x_next_norm - x_min) / x_scale
    return x_next.detach().cpu().numpy()


@dataclass
class Linearization:
    A: np.ndarray
    B: np.ndarray
    c: np.ndarray


def linearize_surrogate(model: CNNLSTM, scaler_X, scaler_U, x_bar, u_bar, device: torch.device) -> Linearization:
    """Vectorized Jacobian-based linearization to reduce autograd overhead."""
    x0 = torch.as_tensor(np.array(x_bar, dtype=np.float32, copy=True), dtype=torch.float32, device=device).view(-1)
    u0 = torch.as_tensor(np.array(u_bar, dtype=np.float32, copy=True), dtype=torch.float32, device=device).view(-1)

    x_scale, x_min = _scaler_params_to_torch(scaler_X, device)
    u_scale, u_min = _scaler_params_to_torch(scaler_U, device)

    nx = x0.shape[0]
    nu = u0.shape[0]

    def model_fn(z: torch.Tensor) -> torch.Tensor:
        x_vec = z[:nx].unsqueeze(0)
        u_vec = z[nx:].unsqueeze(0)
        x_norm = x_vec * x_scale + x_min
        u_norm = u_vec * u_scale + u_min
        model_in = torch.cat([x_norm, u_norm], dim=1).unsqueeze(1)
        x_next_norm = model(model_in).squeeze(0)
        return (x_next_norm - x_min) / x_scale

    z0 = torch.cat([x0, u0]).requires_grad_(True)
    use_cuda_jacobian = device.type == 'cuda'
    use_vectorized_jacobian = not use_cuda_jacobian
    # CUDA + cuDNN RNN backward cannot use eval-mode cuDNN graphs or batched VJP.
    with torch.backends.cudnn.flags(enabled=not use_cuda_jacobian):
        x_next = model_fn(z0)
        J = torch.autograd.functional.jacobian(model_fn, z0, vectorize=use_vectorized_jacobian)

    A = J[:, :nx].detach()
    B = J[:, nx:].detach()
    c = (x_next.detach() - A @ x0.detach() - B @ u0.detach()).detach()
    return Linearization(A=A.cpu().numpy(), B=B.cpu().numpy(), c=c.cpu().numpy())


In [ ]:
# Column definitions + helper utilities
AC_STATE_COLS = [
    'Zone 1 Temperature', 'Zone 2 Temperature', 'Zone 3 Temperature', 'Zone 4 Temperature', 'Zone 5 Temperature'
]
AC_INPUT_COLS = [
    'OutdoorTemperatureWindow',
    'FCU-01 Supply Air Temp',
    'FCU-02 Supply Air Temp - 1 min',
    'FCU-03 Supply Air Temp',
    'FCU-04 Supply Air Temp',
    'FCU-05 Supply Air Temp',
    'PFCU-01 Supply Air Temp',
    'PFCU-02 Supply Air Temp',
    'rain_status',
    'Solar Radiation',
]
NV_INPUT_COLS_BASE = [
    'OutdoorTemperatureWindow',
    'Wind Speed',
    'Wind Direction',
    'rain_status',
    'Solar Radiation',
]
NV_INPUT_COLS = [
    'OutdoorTemperatureWindow', 'Wind Speed', 'Wind Direction', 'PFCU-01 Supply Air Temp', 'PFCU-02 Supply Air Temp', 'rain_status', 'Solar Radiation'
]
COLUMNS_TO_CHECK = list(set(AC_STATE_COLS + AC_INPUT_COLS + NV_INPUT_COLS + ['Z1 Windows Open Close Status']))

# Scalar control drives all FCU/PFCU supply temperatures in each regime
AC_CONTROL_IDX = [1, 2, 3, 4, 5, 6, 7]
NV_CONTROL_IDX = [3, 4]

# Training split definitions (match ac_train_cnn_lstm.ipynb / nv_train_cnn_lstm.ipynb)
AC_TRAIN_COLUMNS = [
    'Zone 1 Temperature', 'Zone 2 Temperature', 'Zone 3 Temperature', 'Zone 4 Temperature', 'Zone 5 Temperature',
    'OutdoorTemperatureWindow', 'FCU-01 Supply Air Temp', 'FCU-02 Supply Air Temp - 1 min', 'FCU-03 Supply Air Temp',
    'FCU-04 Supply Air Temp', 'FCU-05 Supply Air Temp', 'PFCU-01 Supply Air Temp', 'PFCU-02 Supply Air Temp',
    'rain_status', 'Solar Radiation'
]
NV_TRAIN_COLUMNS = [
    'Zone 1 Temperature', 'Zone 2 Temperature', 'Zone 3 Temperature', 'Zone 4 Temperature', 'Zone 5 Temperature',
    'OutdoorTemperatureWindow', 'Wind Speed', 'Wind Direction', 'PFCU-01 Supply Air Temp', 'PFCU-02 Supply Air Temp',
    'rain_status', 'Solar Radiation'
]
DATES_TO_REMOVE = pd.to_datetime(['2024-10-31', '2024-11-04'])
SPLIT_RATIOS = (0.7, 0.15, 0.15)
_SPLIT_DATES_CACHE = {}


def _pfcus_for_nv(row: pd.Series, supply: np.ndarray) -> tuple[float, float]:
    # Extract PFCU-01/02 supply temps for NV/PFC mode from a control vector.
    supply_arr = np.asarray(supply, dtype=float).reshape(-1)
    if supply_arr.shape[0] != 2:
        raise ValueError(f'NV/PFC supply must have 2 entries (PFCU-01, PFCU-02), got {supply_arr.shape[0]}')
    p1 = float(supply_arr[0])
    p2 = float(supply_arr[1])
    return p1, p2


def build_u_vector(row: pd.Series, mode: str, supply) -> np.ndarray:
    # Map control vector into surrogate input (AC: 6 entries, NV/PFC: 2 entries).
    if mode == 'ac':
        supply_arr = np.asarray(supply, dtype=float).reshape(-1)
        if supply_arr.shape[0] != len(AC_CONTROL_IDX):
            raise ValueError(f'AC supply must have {len(AC_CONTROL_IDX)} entries, got {supply_arr.shape[0]}')
        u_vec = row[AC_INPUT_COLS].astype(float).to_numpy(copy=True)
        u_vec[AC_CONTROL_IDX] = supply_arr
    else:
        p1, p2 = _pfcus_for_nv(row, supply)
        u_vec = row[NV_INPUT_COLS_BASE].astype(float).to_list()
        u_vec.insert(3, p1)
        u_vec.insert(4, p2)
        u_vec = np.asarray(u_vec, dtype=float)
    return u_vec


def lift_linearization(A1: np.ndarray, B1: np.ndarray, c1: np.ndarray, steps: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    'Accumulate 1-min linear model over `steps` minutes with hold control.'
    A_acc = np.linalg.matrix_power(A1, steps)
    B_acc = np.zeros_like(B1)
    c_acc = np.zeros_like(c1)
    A_power = np.eye(A1.shape[0])
    for _ in range(steps):
        B_acc += A_power @ B1
        c_acc += A_power @ c1
        A_power = A_power @ A1
    return A_acc, B_acc, c_acc


def make_continuous_segments(df: pd.DataFrame) -> List[pd.DataFrame]:
    list_of_dfs = []
    current_df = [df.iloc[0]]
    for i in range(1, len(df)):
        if (df.iloc[i]['date'] - df.iloc[i-1]['date']).seconds == 60:
            current_df.append(df.iloc[i])
        else:
            list_of_dfs.append(pd.DataFrame(current_df))
            current_df = [df.iloc[i]]
    list_of_dfs.append(pd.DataFrame(current_df))
    return list_of_dfs


def _load_filtered_data(data_path: str, mode_value: Optional[int] = None) -> pd.DataFrame:
    df = pd.read_csv(data_path)
    if mode_value is not None:
        df = df[df['Z1 Windows Open Close Status'] == mode_value]
    df['date'] = pd.to_datetime(df['date'])
    df = df[
        (df['date'].dt.time >= pd.to_datetime('07:30').time()) &
        (df['date'].dt.time <= pd.to_datetime('19:00').time())
    ]
    df = df[df['date'].dt.weekday < 5]
    df = df[~df['date'].dt.normalize().isin(DATES_TO_REMOVE)]
    df = df.sort_values(by='date').reset_index(drop=True)
    if 'Solar Radiation' in df.columns:
        df.loc[:, 'Solar Radiation'] = df['Solar Radiation'].interpolate(method='nearest', limit_direction='both')
    return df


def _split_segment_dates(df: pd.DataFrame, columns_to_check, min_len: int = 10) -> Dict[str, List[str]]:
    segments = [seg.reset_index(drop=True) for seg in make_continuous_segments(df) if len(seg) >= min_len]
    clean_segments = [
        seg.dropna(subset=columns_to_check)
        for seg in segments
        if not seg[columns_to_check].isna().any().any()
    ]
    dates = [seg['date'].iloc[0].date().isoformat() for seg in clean_segments]
    split_ratio, val_ratio, _ = SPLIT_RATIOS
    split_index = int(len(clean_segments) * split_ratio)
    val_index = int(len(clean_segments) * (split_ratio + val_ratio))
    return {
        'train': dates[:split_index],
        'val': dates[split_index:val_index],
        'test': dates[val_index:],
    }


def get_split_dates(data_path: str, policy: str = 'intersection') -> Dict[str, List[str]]:
    cache_key = (data_path, policy)
    if cache_key in _SPLIT_DATES_CACHE:
        return _SPLIT_DATES_CACHE[cache_key]

    ac_df = _load_filtered_data(data_path, mode_value=0)
    nv_df = _load_filtered_data(data_path, mode_value=1)
    ac_dates = _split_segment_dates(ac_df, AC_TRAIN_COLUMNS)
    nv_dates = _split_segment_dates(nv_df, NV_TRAIN_COLUMNS)

    if policy == 'intersection':
        combined = {k: sorted(set(ac_dates[k]) & set(nv_dates[k])) for k in ac_dates}
    elif policy == 'union':
        combined = {k: sorted(set(ac_dates[k]) | set(nv_dates[k])) for k in ac_dates}
    elif policy == 'ac':
        combined = {k: sorted(set(ac_dates[k])) for k in ac_dates}
    elif policy == 'nv':
        combined = {k: sorted(set(nv_dates[k])) for k in ac_dates}
    else:
        raise ValueError(f'Unknown test date policy: {policy}')

    _SPLIT_DATES_CACHE[cache_key] = combined
    return combined


def _normalize_segment_split(split: Optional[str]) -> str:
    key = str(split or 'all').strip().lower().replace(' ', '')
    aliases = {
        'train': 'train',
        'val': 'val',
        'valid': 'val',
        'validate': 'val',
        'validation': 'val',
        'test': 'test',
        'test+val': 'test+val',
        'val+test': 'test+val',
        'test_val': 'test+val',
        'val_test': 'test+val',
        'test,val': 'test+val',
        'val,test': 'test+val',
        'all': 'all',
        '*': 'all',
        'none': 'all',
        '': 'all',
    }
    if key not in aliases:
        raise ValueError(
            "Unknown split. Use one of: 'test', 'val'/'validate', 'test+val', 'train', or 'all'. "
            f"Got: {split!r}"
        )
    return aliases[key]


def prepare_segments(
    data_path: str,
    max_segments: int = 40,
    split: str = 'test',
    test_policy: str = 'intersection',
    min_len: int = 600,
) -> List[pd.DataFrame]:
    df = _load_filtered_data(data_path)
    segments = [seg.reset_index(drop=True) for seg in make_continuous_segments(df) if len(seg) >= min_len]
    clean_segments = [
        seg.dropna(subset=COLUMNS_TO_CHECK)
        for seg in segments
        if not seg[COLUMNS_TO_CHECK].isna().any().any()
    ]

    split_key = _normalize_segment_split(split)
    if split_key != 'all':
        split_dates = get_split_dates(data_path, policy=test_policy)
        if split_key == 'test+val':
            split_set = set(split_dates.get('test', [])) | set(split_dates.get('val', []))
        else:
            split_set = set(split_dates.get(split_key, []))
        clean_segments = [
            seg for seg in clean_segments
            if seg['date'].iloc[0].date().isoformat() in split_set
        ]

    return clean_segments[:max_segments]


In [ ]:
# MPC core (supports multi-state)
try:
    import gurobipy as gp
    from gurobipy import GRB
    GUROBI_OK = True
except ImportError:
    GUROBI_OK = False


SPECIFIC_HEAT_AIR = 1005.0  # J/(kg*K)
AIR_DENSITY = 1.225         # kg/m^3
FCU_AIRFLOW_CFM = 800.0     # ft^3/min per FCU terminal, constant-volume assumption
LOCAL_COOLING_CFM = 1000.0  # ft^3/min per PFCU terminal, constant-volume assumption
CFM_TO_M3S = 0.0283168 / 60.0
JOULES_PER_KWH = 3.6e6
DEFAULT_COP_FCU = 5.0
DEFAULT_COP_PFC = 5.0

# PMV regression coefficients from temp_rh_pmv_regression_summary.csv
PMV_LINEAR_COEFFS = {
    1: (-7.673842, 0.249984, 0.011084),
    2: (-7.539536, 0.244956, 0.011203),
    3: (-7.595545, 0.245508, 0.011841),
    4: (-7.625768, 0.246251, 0.011946),
    5: (-7.599742, 0.246740, 0.011352),
}

PMV_QUAD_COEFFS = {
    1: (-2.668230, 0.027159, -0.043495, 0.001987, 0.000063, 0.001618),
    2: (-2.952313, 0.044535, -0.042467, 0.001691, 0.000056, 0.001615),
    3: (-3.600209, 0.097232, -0.046012, 0.000616, 0.000051, 0.001761),
    4: (-2.617966, 0.025267, -0.044229, 0.001944, 0.000056, 0.001682),
    5: (-2.581557, 0.022383, -0.044050, 0.001940, 0.000046, 0.001725),
}


def _extract_outdoor_rh(exog_row: pd.Series, rh_nv_source_col: str = "OutdoorHumidityWindow") -> float:
    return float(
        exog_row.get(
            rh_nv_source_col,
            exog_row.get("OutdoorHumidityWindow", exog_row.get("Outdoor Humidity", exog_row.get("OutdoorHumidity", 65.0))),
        )
    )


def _rain_active(value, threshold: float) -> bool:
    try:
        rain_value = float(value)
    except (TypeError, ValueError):
        rain_value = 0.0
    if not np.isfinite(rain_value):
        rain_value = 0.0
    return rain_value >= float(threshold)


def current_observed_rain_active(cfg, current_exog_row: pd.Series) -> bool:
    if not bool(getattr(cfg, "enforce_current_rain_safety", False)):
        return False
    threshold = float(getattr(cfg, "rain_active_threshold", 0.5))
    return _rain_active(current_exog_row.get("rain_status", 0.0), threshold)


def rain_lockout_steps(cfg) -> int:
    lockout_min = max(int(getattr(cfg, "rain_lockout_min", 0)), 0)
    ctrl_period_min = max(int(getattr(cfg, "ctrl_period_min", 5)), 1)
    return int(np.ceil(lockout_min / ctrl_period_min)) if lockout_min > 0 else 0


def observed_rain_forces_ac(cfg, current_exog_row: pd.Series, rain_lock_steps: int = 0) -> bool:
    """Return whether the current receding-horizon move must be AC because of observed rain or lockout."""
    if not bool(getattr(cfg, "enforce_current_rain_safety", False)):
        return False
    return current_observed_rain_active(cfg, current_exog_row) or int(rain_lock_steps) > 0


def update_observed_rain_lock_steps(cfg, current_exog_row: pd.Series, previous_lock_steps: int) -> int:
    """Advance the 30-minute observed-rain lockout at one MPC control horizon."""
    if not bool(getattr(cfg, "enforce_current_rain_safety", False)):
        return 0
    if current_observed_rain_active(cfg, current_exog_row):
        return rain_lockout_steps(cfg)
    return max(int(previous_lock_steps) - 1, 0)


def neutralize_rain_for_mpc_prediction(cfg, exog_5min_forecast: pd.DataFrame) -> pd.DataFrame:
    """Return a forecast copy with rain_status set to no-rain for MIQP prediction only."""
    if not bool(getattr(cfg, "ignore_rain_in_mpc_prediction", False)):
        return exog_5min_forecast
    if "rain_status" not in exog_5min_forecast.columns:
        return exog_5min_forecast
    forecast = exog_5min_forecast.copy()
    forecast["rain_status"] = 0.0
    return forecast


def _pmv_expr(zone_id: int, temp_expr, rh_value: float, model: str):
    model = str(model).lower()
    if model == "linear":
        b0, bt, brh = PMV_LINEAR_COEFFS[int(zone_id)]
        return b0 + bt * temp_expr + brh * float(rh_value)
    if model == "second_order":
        b0, bt, brh, bt2, brh2, btrh = PMV_QUAD_COEFFS[int(zone_id)]
        rh = float(rh_value)
        return b0 + bt * temp_expr + brh * rh + bt2 * temp_expr * temp_expr + brh2 * rh * rh + btrh * temp_expr * rh
    raise ValueError(f"Unsupported pmv_model={model!r}; expected 'linear' or 'second_order'.")


class RepoHooks:
    def load_day_dataframe(self, day_idx: int):
        raise NotImplementedError

    def plant_step_1min(self, mode: int, x, u, exog_row: pd.Series):
        raise NotImplementedError

    def linearize_5min(self, mode: int, x_oper, u_oper, exog_fore_5min: pd.Series):
        raise NotImplementedError


@dataclass
class MPCConfig:
    # Comfort targets
    T_ref_ac: float = 27.0
    T_nv_max: float = 31.0
    T_nv_soft: float = 28.5
    dt_min: int = 1
    ctrl_period_min: int = 5
    horizon_min: int = 60
    x_min: float = 22.0
    x_max: float = 30.0
    u_min: float = 12.0  # legacy lower bound (kept for compatibility)
    u_min_ac: float = 12.5
    u_min_pfc: float = 18.5
    u_max_ac: float = 21.0
    u_max_pfc: float = 18.5
    w_du: float = 1e1
    w_switch: float = 1e1
    w_energy: float = 1e4
    pmv_target: float = -0.5
    pmv_model: str = "linear"  # options: "linear", "second_order"
    rh_ac_fixed: float = 65.0
    rh_nv_source_col: str = "OutdoorHumidityWindow"
    enforce_current_rain_safety: bool = False
    rain_active_threshold: float = 0.5
    rain_lockout_min: int = 0
    ignore_rain_in_mpc_prediction: bool = False
    pmv_big_m: float = 8.0  # used by second_order PMV gating
    energy_norm_kwh: float = 1.0
    pfc_big_m: float = 50.0
    min_dwell_steps: int = 0  # 0 disables dwell-time lock
    energy_coeff_scale: float = 1.0
    cop_fcu: float = DEFAULT_COP_FCU
    cop_pfc: float = DEFAULT_COP_PFC
    linearize_energy: bool = False
    penalize_undercool: bool = False
    use_pv_objective: bool = False
    pv_model: str = "simple_linear"  # options: "simple_linear", "appendix_b"
    pv_alpha: float = 0.002
    pv_pmax: Optional[float] = 3.0
    pv_a1: float = 0.0
    pv_a2: float = 0.0
    pv_a3: float = 0.0
    pv_source_area_sqft: float = 5000.0
    building_area_sqft: float = 75.0
    pv_scale_factor: Optional[float] = None
    w_pv_export: float = 1.0
    price_low: float = 1.0
    price_high: float = 4.0
    T_out_nv_ref: float = 30.0
    w_terminal: float = 1.0
    soften_x_bounds: bool = True
    w_x_slack: float = 1e2
    mip_time_limit: Optional[float] = 600
    gurobi_output: int = 0
    gurobi_presolve: Optional[int] = 1
    gurobi_mipfocus: Optional[int] = 2
    gurobi_heuristics: Optional[float] = 0.1
    gurobi_nonconvex: Optional[int] = None
    write_iis_on_infeasible: bool = True


def time_in_windows(ts: pd.Timestamp) -> bool:
    h, m = ts.hour, ts.minute
    minutes = h * 60 + m
    w1 = (7*60 + 30) <= minutes < (10*60)
    w2 = (16*60 + 30) <= minutes < (19*60)
    return w1 or w2

def _pv_scale(cfg: MPCConfig) -> float:
    explicit_scale = getattr(cfg, "pv_scale_factor", None)
    if explicit_scale is not None:
        return float(explicit_scale)

    source_area = float(getattr(cfg, "pv_source_area_sqft", 0.0) or 0.0)
    building_area = float(getattr(cfg, "building_area_sqft", 0.0) or 0.0)
    if source_area > 0 and building_area > 0:
        return building_area / source_area
    return 1.0

def pv_available_power_kw(cfg: MPCConfig, I_solar: float, T_out: float) -> float:
    ghi = max(float(I_solar), 0.0)
    t_out = float(T_out)
    pv_model = str(getattr(cfg, "pv_model", "simple_linear")).lower()
    scale = _pv_scale(cfg)

    if pv_model == "appendix_b":
        pv_kw = scale * (cfg.pv_a1 * ghi * ghi + cfg.pv_a2 * ghi * t_out + cfg.pv_a3 * ghi)
    elif pv_model == "simple_linear":
        pv_kw = scale * (cfg.pv_alpha * ghi)
    else:
        raise ValueError(f"Unsupported pv_model={cfg.pv_model!r}; expected 'simple_linear' or 'appendix_b'.")

    pv_kw = max(pv_kw, 0.0)
    pv_pmax = getattr(cfg, "pv_pmax", None)
    if pv_pmax is not None:
        pv_kw = min(pv_kw, float(pv_pmax))
    return pv_kw

def _validate_cop(name: str, value: float) -> float:
    value = float(value)
    if not np.isfinite(value) or value <= 0.0:
        raise ValueError(f"{name} must be finite and > 0; got {value!r}")
    return value


def hvac_power_heat_exchange_kw(
    cfg: MPCConfig,
    z_mode: int,
    zone_temps,
    fcu_supplies,
    pfc_supplies,
    T_out: float,
) -> float:
    """Electrical cooling power from constant-volume heat exchange; fan power excluded."""
    cop_fcu = _validate_cop("cop_fcu", getattr(cfg, "cop_fcu", DEFAULT_COP_FCU))
    cop_pfc = _validate_cop("cop_pfc", getattr(cfg, "cop_pfc", DEFAULT_COP_PFC))
    fcu_coeff_kw_per_k = SPECIFIC_HEAT_AIR * AIR_DENSITY * (FCU_AIRFLOW_CFM * CFM_TO_M3S) / 1000.0 / cop_fcu
    pfc_coeff_kw_per_k = SPECIFIC_HEAT_AIR * AIR_DENSITY * (LOCAL_COOLING_CFM * CFM_TO_M3S) / 1000.0 / cop_pfc

    if int(round(float(z_mode))) == 1:
        x = np.asarray(zone_temps, dtype=float)
        u = np.asarray(fcu_supplies, dtype=float)
        delta = np.maximum(x - u, 0.0)
        return float(fcu_coeff_kw_per_k * np.nan_to_num(delta, nan=0.0).sum())

    u_pfc = np.asarray(pfc_supplies, dtype=float)
    delta = np.maximum(float(T_out) - u_pfc, 0.0)
    return float(pfc_coeff_kw_per_k * np.nan_to_num(delta, nan=0.0).sum())

def _explain_mpc_decision(step_idx: int, total_steps: int, t: pd.Timestamp, debug: Dict[str, object]):
    if debug is None:
        return

    z0 = int(debug.get('z0', 0))
    mode = 'AC' if z0 == 1 else 'NV'
    k0 = debug.get('k0', {})
    breakdown = debug.get('breakdown', {})

    print(f"\n[MPC-Explain] step {step_idx + 1}/{total_steps} @ {t}")
    print(f"  chosen mode: {mode} (z0={z0}), obj={debug.get('obj', float('nan')):.3f}")

    cf = debug.get('counterfactual_z0', None)
    if isinstance(cf, dict):
        ac = cf.get('ac', {})
        nv = cf.get('nv', {})

        def _fmt3(v):
            try:
                return f"{float(v):.3f}"
            except Exception:
                return "nan"

        def _fmt_breakdown(bd):
            if not isinstance(bd, dict):
                return "n/a"
            return (
                f"comfort={_fmt3(bd.get('comfort'))}, "
                f"du={_fmt3(bd.get('du'))}, "
                f"switch={_fmt3(bd.get('switch'))}, "
                f"energy={_fmt3(bd.get('energy'))}, "
                f"x_slack={_fmt3(bd.get('x_slack'))}, "
                f"pv={_fmt3(bd.get('pv'))}"
            )

        print(
            "  forced z0 counterfactual: "
            f"obj(z0=AC)={_fmt3(ac.get('obj'))}, "
            f"obj(z0=NV)={_fmt3(nv.get('obj'))}, "
            f"x1_mean(z0=AC)={_fmt3(ac.get('x1_mean'))}, "
            f"x1_mean(z0=NV)={_fmt3(nv.get('x1_mean'))}"
        )
        print(f"    breakdown(z0=AC): {_fmt_breakdown(ac.get('breakdown'))}")
        print(f"    breakdown(z0=NV): {_fmt_breakdown(nv.get('breakdown'))}")
    elif debug.get('counterfactual_z0_error'):
        print(f"  forced z0 counterfactual unavailable: {debug.get('counterfactual_z0_error')}")

    print(
        "  k=0 signals: "
        f"T_out={k0.get('T_out', float('nan')):.2f}, "
        f"rh_nv={k0.get('rh_nv', float('nan')):.2f}, "
        f"sw0={k0.get('sw0', float('nan')):.3f}, "
        f"pmv_target={k0.get('pmv_target', float('nan')):.2f}, "
        f"pmv_ac_dev_sq_sum={k0.get('pmv_ac_dev_sq_sum', k0.get('pmv_ac_sq_sum', float('nan'))):.4f}, "
        f"pmv_nv_dev_sq_sum={k0.get('pmv_nv_dev_sq_sum', k0.get('pmv_nv_sq_sum', float('nan'))):.4f}, "
        f"x_high_sum={k0.get('x_high_sum', float('nan')):.4f}"
    )
    print(
        "  k=0 energy proxies (kWh per control stage): "
        f"AC_like={k0.get('energy_fcu0_kwh', float('nan')):.5f}, "
        f"NV_like={k0.get('energy_pfc0_kwh', float('nan')):.5f}, "
        f"used={k0.get('energy_used0_kwh', float('nan')):.5f}"
    )

    t_ac = k0.get('x_next_mean_ac0')
    t_nv = k0.get('x_next_mean_nv0')
    if t_ac is not None and t_nv is not None:
        print(f"  one-step mean temp forecast: AC={t_ac:.3f}, NV={t_nv:.3f}")

    pmv_ac_metric = float(k0.get('pmv_ac_dev_sq_sum', k0.get('pmv_ac_sq_sum', 1e9)))
    pmv_nv_metric = float(k0.get('pmv_nv_dev_sq_sum', k0.get('pmv_nv_sq_sum', 1e9)))

    reasons = []
    if z0 == 0:
        if pmv_nv_metric <= pmv_ac_metric:
            reasons.append('NV PMV deviation^2 from target at k=0 is no worse than AC.')
        if k0.get('energy_pfc0_kwh', 1e9) < k0.get('energy_fcu0_kwh', 1e9):
            reasons.append('NV-like stage energy is lower than AC-like stage energy.')
    else:
        if pmv_ac_metric < pmv_nv_metric:
            reasons.append('AC PMV deviation^2 from target at k=0 is better than NV.')
        if k0.get('x_high_sum', 0.0) > 1e-6:
            reasons.append('Global x_max soft-bound pressure is active at k=0.')
        if k0.get('energy_fcu0_kwh', 0.0) <= k0.get('energy_pfc0_kwh', 0.0):
            reasons.append('AC-like stage energy is not worse than NV-like at k=0.')

    if reasons:
        print('  inferred drivers:')
        for r in reasons:
            print('   -', r)

    print(
        "  objective breakdown: "
        f"comfort={breakdown.get('comfort', float('nan')):.3f}, "
        f"du={breakdown.get('du', float('nan')):.3f}, "
        f"switch={breakdown.get('switch', float('nan')):.3f}, "
        f"energy={breakdown.get('energy', float('nan')):.3f}, "
        f"x_slack={breakdown.get('x_slack', float('nan')):.3f}, "
        f"pv={breakdown.get('pv', float('nan')):.3f}"
    )


def solve_miqp(cfg: MPCConfig, x0, u_prev: float, z_prev: int, t0: pd.Timestamp, exog_5min_forecast: pd.DataFrame, hooks: RepoHooks, lock_steps: int = 0, verbose: bool = False, force_z0: Optional[int] = None, rain_force_ac_now: bool = False, collect_diagnostics: bool = False) -> Dict[str, float]:
    if not GUROBI_OK:
        raise RuntimeError("Gurobi not available. Install gurobipy to run the MIQP.")

    # Negative comfort/energy/slack penalties can create pathological or unbounded objectives.
    nonneg_weight_fields = (
        "w_du",
        "w_switch",
        "w_energy",
        "w_x_slack",
        "w_terminal",
        "w_pv_export",
    )
    for name in nonneg_weight_fields:
        value = float(getattr(cfg, name, 0.0))
        if not np.isfinite(value):
            raise ValueError(f"{name} must be finite; got {value!r}")
        if value < 0.0:
            raise ValueError(f"{name} must be >= 0; got {value}")

    x0_vec = np.asarray(x0, dtype=float).reshape(-1)
    nx = x0_vec.shape[0]
    H = len(exog_5min_forecast)
    u_prev_vec = np.asarray(u_prev, dtype=float).reshape(-1)
    rain_force_ac_now = bool(rain_force_ac_now) and bool(getattr(cfg, "enforce_current_rain_safety", False))

    # Keep linearization horizon aligned with control period if hooks expose steps_per_ctrl.
    if hasattr(hooks, "steps_per_ctrl"):
        steps_per_ctrl = int(getattr(hooks, "steps_per_ctrl"))
        if steps_per_ctrl != cfg.ctrl_period_min:
            hooks.steps_per_ctrl = int(cfg.ctrl_period_min)
            if hasattr(hooks, "reset_cache"):
                hooks.reset_cache()

    linearize_energy = bool(getattr(cfg, "linearize_energy", False))
    pmv_model = str(getattr(cfg, "pmv_model", "linear")).lower()
    if pmv_model not in ("linear", "second_order"):
        raise ValueError(f"Unsupported pmv_model={pmv_model!r}; expected 'linear' or 'second_order'.")

    m = gp.Model("mmv_mpc")
    m.Params.OutputFlag = int(getattr(cfg, "gurobi_output", 0))

    nonconvex = getattr(cfg, "gurobi_nonconvex", None)
    if (not linearize_energy or pmv_model == "second_order") and nonconvex is None:
        nonconvex = 2
    if nonconvex is not None:
        m.Params.NonConvex = int(nonconvex)

    presolve = getattr(cfg, "gurobi_presolve", 1)
    if presolve is not None:
        m.Params.Presolve = int(presolve)
    mipfocus = getattr(cfg, "gurobi_mipfocus", 1)
    if mipfocus is not None:
        m.Params.MIPFocus = int(mipfocus)
    heuristics = getattr(cfg, "gurobi_heuristics", 0.1)
    if heuristics is not None:
        m.Params.Heuristics = float(heuristics)

    if cfg.mip_time_limit:
        m.Params.TimeLimit = float(cfg.mip_time_limit)

    soften_x_bounds = bool(getattr(cfg, "soften_x_bounds", True))
    w_x_slack = float(getattr(cfg, "w_x_slack", 1e4))

    ctrl_count = len(AC_CONTROL_IDX)
    fcu_indices = [0, 1, 2, 3, 4]
    pfc_indices = [5, 6]
    pfc_dataset_cols = ["PFCU-01 Supply Air Temp", "PFCU-02 Supply Air Temp"]
    t_out_fore = [float(exog_5min_forecast.iloc[k].get("T_out", exog_5min_forecast.iloc[k].get("OutdoorTemperatureWindow", 0.0))) for k in range(H)]
    rh_nv_fore = [_extract_outdoor_rh(exog_5min_forecast.iloc[k], getattr(cfg, "rh_nv_source_col", "OutdoorHumidityWindow")) for k in range(H)]
    rh_nv_fore_h = rh_nv_fore + ([rh_nv_fore[-1]] if H > 0 else [float(getattr(cfg, "rh_ac_fixed", 65.0))])
    pfc_data_fore = {
        p_idx: [float(exog_5min_forecast.iloc[k].get(col, t_out_fore[k])) for k in range(H)]
        for p_idx, col in enumerate(pfc_dataset_cols)
    }
    u_lower_bound = {}
    u_upper_bound = {}
    for k in range(H):
        for j in fcu_indices:
            u_lower_bound[(k, j)] = cfg.u_min_ac
            u_upper_bound[(k, j)] = cfg.u_max_ac
        for p_idx, j in enumerate(pfc_indices):
            t_out_k = t_out_fore[k]
            t_pfc_data_k = pfc_data_fore[p_idx][k]
            u_lower_bound[(k, j)] = min(cfg.u_min_pfc, t_out_k, t_pfc_data_k)
            u_upper_bound[(k, j)] = max(cfg.u_max_pfc, t_out_k, t_pfc_data_k)

    x = m.addVars(H + 1, nx, name="x")
    u = m.addVars(H, ctrl_count, lb=u_lower_bound, ub=u_upper_bound, name="u")
    z_mode = m.addVars(H, vtype=GRB.BINARY, name="z_mode")
    if force_z0 is not None:
        force_z0 = int(force_z0)
        if force_z0 not in (0, 1):
            raise ValueError(f"force_z0 must be 0, 1, or None; got {force_z0!r}")
        if H <= 0:
            raise ValueError("force_z0 requires a non-empty horizon")
        if rain_force_ac_now and force_z0 != 1:
            raise ValueError("rain_force_ac_now conflicts with force_z0=0")
        m.addConstr(z_mode[0] == force_z0, name="force_z0")
    if rain_force_ac_now:
        if H <= 0:
            raise ValueError("rain_force_ac_now requires a non-empty horizon")
        m.addConstr(z_mode[0] == 1, name="current_rain_force_ac")
    pfc_on = m.addVars(H, vtype=GRB.BINARY, name="pfc_on")
    e_pos_ac = m.addVars(H + 1, nx, lb=0.0, name="e_pos_ac")
    e_neg_ac = m.addVars(H + 1, nx, lb=0.0, name="e_neg_ac")
    s_nv_high = m.addVars(H + 1, nx, lb=0.0, name="s_nv_high")
    s_nv_soft = m.addVars(H + 1, nx, lb=0.0, name="s_nv_soft")
    pmv_ac = m.addVars(H + 1, nx, lb=-GRB.INFINITY, name="pmv_ac")
    pmv_nv = m.addVars(H + 1, nx, lb=-GRB.INFINITY, name="pmv_nv")
    sw = m.addVars(H, lb=0.0, name="sw")
    lift_fcu_raw = m.addVars(H, nx, lb=-GRB.INFINITY, name="lift_fcu_raw")
    lift_pfc_raw = m.addVars(H, len(pfc_indices), lb=-GRB.INFINITY, name="lift_pfc_raw")
    delta_cool_fcu = m.addVars(H, nx, lb=0.0, name="delta_cool_fcu")
    delta_cool_pfc = m.addVars(H, len(pfc_indices), lb=0.0, name="delta_cool_pfc")
    energy_fcu_used = m.addVars(H, lb=0.0, name="energy_fcu_used") if linearize_energy else None
    energy_pfc_used = m.addVars(H, lb=0.0, name="energy_pfc_used") if linearize_energy else None

    s_x_low = m.addVars(H + 1, nx, lb=0.0, name="s_x_low")
    s_x_high = m.addVars(H + 1, nx, lb=0.0, name="s_x_high")

    step_seconds = cfg.ctrl_period_min * 60
    energy_coeff_scale = float(getattr(cfg, "energy_coeff_scale", 1.0))
    if energy_coeff_scale <= 0:
        raise ValueError("energy_coeff_scale must be > 0")
    cop_fcu = _validate_cop("cop_fcu", getattr(cfg, "cop_fcu", DEFAULT_COP_FCU))
    cop_pfc = _validate_cop("cop_pfc", getattr(cfg, "cop_pfc", DEFAULT_COP_PFC))
    dt_h = step_seconds / 3600.0
    # Constant-volume heat exchange. Flow constants are per terminal unit, so the
    # power/energy expressions multiply by the sum of each unit's temperature lift.
    # Fan power is excluded by design for this constant-volume formulation.
    fcu_power_coeff = SPECIFIC_HEAT_AIR * AIR_DENSITY * (FCU_AIRFLOW_CFM * CFM_TO_M3S) / 1000.0 / cop_fcu
    local_power_coeff = SPECIFIC_HEAT_AIR * AIR_DENSITY * (LOCAL_COOLING_CFM * CFM_TO_M3S) / 1000.0 / cop_pfc
    fcu_energy_coeff = fcu_power_coeff * dt_h * energy_coeff_scale
    local_energy_coeff = local_power_coeff * dt_h * energy_coeff_scale
    w_energy = float(cfg.w_energy) / energy_coeff_scale
    energy_norm_kwh = float(getattr(cfg, "energy_norm_kwh", 1.0))
    if energy_norm_kwh <= 0:
        raise ValueError("energy_norm_kwh must be > 0")
    pfc_big_m = float(getattr(cfg, "pfc_big_m", 50.0))

    if cfg.use_pv_objective:
        p_hvac = m.addVars(H, lb=0.0, name="p_hvac")
        p_grid = m.addVars(H, lb=0.0, name="p_grid")
        p_export = m.addVars(H, lb=0.0, name="p_export")

    for j in range(nx):
        m.addConstr(x[0, j] == float(x0_vec[j]))

    for k in range(H + 1):
        for j in range(nx):
            if soften_x_bounds:
                m.addConstr(x[k, j] + s_x_low[k, j] >= cfg.x_min)
                m.addConstr(x[k, j] - s_x_high[k, j] <= cfg.x_max)
            else:
                m.addConstr(x[k, j] >= cfg.x_min)
                m.addConstr(x[k, j] <= cfg.x_max)

    for k in range(H + 1):
        z_k = z_mode[k] if k < H else z_mode[H - 1]
        rh_nv_k = float(rh_nv_fore_h[k])
        for j in range(nx):
            # Legacy temperature comfort terms retained for compatibility (not used in objective).
            m.addGenConstrIndicator(z_k, True, e_pos_ac[k, j] >= x[k, j] - cfg.T_ref_ac)
            m.addGenConstrIndicator(z_k, True, e_neg_ac[k, j] >= cfg.T_ref_ac - x[k, j])
            m.addGenConstrIndicator(z_k, False, e_pos_ac[k, j] == 0.0)
            m.addGenConstrIndicator(z_k, False, e_neg_ac[k, j] == 0.0)
            m.addGenConstrIndicator(z_k, False, x[k, j] <= cfg.T_nv_max + s_nv_high[k, j])
            m.addGenConstrIndicator(z_k, False, s_nv_soft[k, j] >= x[k, j] - cfg.T_nv_soft)
            m.addGenConstrIndicator(z_k, True, s_nv_soft[k, j] == 0.0)

            # PMV definitions by mode:
            #   AC mode uses fixed RH (cfg.rh_ac_fixed)
            #   NV mode uses outdoor RH (cfg.rh_nv_source_col from exogenous forecast)
            pmv_expr_ac = _pmv_expr(j + 1, x[k, j], float(getattr(cfg, "rh_ac_fixed", 65.0)), pmv_model)
            pmv_expr_nv = _pmv_expr(j + 1, x[k, j], rh_nv_k, pmv_model)

            if pmv_model == "linear":
                # Exact mode gating in linear PMV mode (avoids big-M induced infeasibility under dwell locks).
                m.addGenConstrIndicator(z_k, True, pmv_ac[k, j] - pmv_expr_ac == 0.0)
                m.addGenConstrIndicator(z_k, False, pmv_ac[k, j] == 0.0)
                m.addGenConstrIndicator(z_k, False, pmv_nv[k, j] - pmv_expr_nv == 0.0)
                m.addGenConstrIndicator(z_k, True, pmv_nv[k, j] == 0.0)
            else:
                pmv_big_m = float(getattr(cfg, "pmv_big_m", 8.0))

                # AC active when z_k == 1
                m.addConstr(pmv_ac[k, j] - pmv_expr_ac <= pmv_big_m * (1.0 - z_k))
                m.addConstr(pmv_expr_ac - pmv_ac[k, j] <= pmv_big_m * (1.0 - z_k))
                m.addConstr(pmv_ac[k, j] <= pmv_big_m * z_k)
                m.addConstr(pmv_ac[k, j] >= -pmv_big_m * z_k)

                # NV active when z_k == 0
                m.addConstr(pmv_nv[k, j] - pmv_expr_nv <= pmv_big_m * z_k)
                m.addConstr(pmv_expr_nv - pmv_nv[k, j] <= pmv_big_m * z_k)
                m.addConstr(pmv_nv[k, j] <= pmv_big_m * (1.0 - z_k))
                m.addConstr(pmv_nv[k, j] >= -pmv_big_m * (1.0 - z_k))

    for k in range(H):
        if k == 0:
            m.addConstr(sw[k] >= z_mode[k] - z_prev)
            m.addConstr(sw[k] >= z_prev - z_mode[k])
        else:
            m.addConstr(sw[k] >= z_mode[k] - z_mode[k - 1])
            m.addConstr(sw[k] >= z_mode[k - 1] - z_mode[k])

    for k in range(H):
        m.addConstr(pfc_on[k] <= 1 - z_mode[k])

    # Optional dwell-time lock (disable with min_dwell_steps=0). Observed rain can override
    # the current move so a prior NV dwell lock cannot keep windows open during rain.
    if lock_steps > 0:
        for k in range(min(lock_steps, H)):
            if rain_force_ac_now and k == 0 and int(z_prev) == 0:
                continue
            m.addConstr(z_mode[k] == z_prev)

    min_dwell_steps = int(getattr(cfg, "min_dwell_steps", 0))
    if min_dwell_steps > 1:
        for k in range(1, H):
            lo = max(1, k - min_dwell_steps + 1)
            m.addConstr(gp.quicksum(sw[i] for i in range(lo, k + 1)) <= 1)

    A0_ac = B0_ac = g0_ac = None
    A0_nv = B0_nv = g0_nv = None

    for k in range(H):
        exog_k = exog_5min_forecast.iloc[k]
        T_out = float(exog_k.get("T_out", exog_k.get("OutdoorTemperatureWindow", 0.0)))
        for j in range(nx):
            m.addConstr(lift_fcu_raw[k, j] == x[k, j] - u[k, fcu_indices[j]])
            m.addGenConstrMax(delta_cool_fcu[k, j], [lift_fcu_raw[k, j]], constant=0.0)
        for p_idx, u_idx in enumerate(pfc_indices):
            m.addConstr(lift_pfc_raw[k, p_idx] == T_out - u[k, u_idx])
            m.addGenConstrMax(delta_cool_pfc[k, p_idx], [lift_pfc_raw[k, p_idx]], constant=0.0)

        # Optional PFCU on/off during NV mode (z_mode=0)
        for p_idx, u_idx in enumerate(pfc_indices):
            T_pfc_data = float(exog_k.get(pfc_dataset_cols[p_idx], T_out))

            # If NV and PFCU is ON, pin supply to dataset real-time value
            m.addConstr(u[k, u_idx] - T_pfc_data <= pfc_big_m * (z_mode[k] + (1 - pfc_on[k])))
            m.addConstr(T_pfc_data - u[k, u_idx] <= pfc_big_m * (z_mode[k] + (1 - pfc_on[k])))
            # If NV and PFCU is OFF, pin supply to outdoor temperature
            m.addConstr(u[k, u_idx] - T_out <= pfc_big_m * (z_mode[k] + pfc_on[k]))
            m.addConstr(T_out - u[k, u_idx] <= pfc_big_m * (z_mode[k] + pfc_on[k]))

            # If AC, pin PFCU supply to fixed setpoint (e.g., 21°C)
            m.addGenConstrIndicator(z_mode[k], True, u[k, u_idx] == cfg.u_min_pfc)

        A_ac, B_ac, g_ac = hooks.linearize_5min(1, x0_vec, u_prev_vec, exog_k)
        A_nv, B_nv, g_nv = hooks.linearize_5min(0, x0_vec, u_prev_vec[-2:], exog_k)
        if k == 0:
            A0_ac, B0_ac, g0_ac = A_ac.copy(), B_ac.copy(), g_ac.copy()
            A0_nv, B0_nv, g0_nv = A_nv.copy(), B_nv.copy(), g_nv.copy()

        for j in range(nx):
            lhs_ac = gp.quicksum(A_ac[j, i] * x[k, i] for i in range(nx)) + gp.quicksum(B_ac[j, l] * u[k, l] for l in range(ctrl_count)) + float(g_ac[j])
            lhs_nv = gp.quicksum(A_nv[j, i] * x[k, i] for i in range(nx)) + gp.quicksum(B_nv[j, idx] * u[k, pfc_indices[idx]] for idx in range(len(pfc_indices))) + float(g_nv[j])
            m.addGenConstrIndicator(z_mode[k], True, x[k+1, j] - lhs_ac == 0.0)
            m.addGenConstrIndicator(z_mode[k], False, x[k+1, j] - lhs_nv == 0.0)

        if cfg.use_pv_objective:
            T_out = float(exog_k.get("T_out", exog_k.get("OutdoorTemperatureWindow", 0.0)))
            fcu_power = fcu_power_coeff * gp.quicksum(delta_cool_fcu[k, j] for j in range(nx))
            pfc_power = local_power_coeff * gp.quicksum(delta_cool_pfc[k, p] for p in range(len(pfc_indices)))
            # Mode-gated electrical cooling power for PV/grid balance.
            # z_mode=1: FCU cooling; z_mode=0: PFCU/local cooling.
            m.addGenConstrIndicator(z_mode[k], True, p_hvac[k] - fcu_power == 0.0)
            m.addGenConstrIndicator(z_mode[k], False, p_hvac[k] - pfc_power == 0.0)

            I_solar = float(exog_k.get("I_solar", exog_k.get("Solar Radiation", 0.0)))
            p_pv = pv_available_power_kw(cfg, I_solar=I_solar, T_out=T_out)
            m.addConstr(p_grid[k] >= p_hvac[k] - p_pv)
            m.addConstr(p_export[k] >= p_pv - p_hvac[k])
            m.addConstr(p_grid[k] >= 0.0)
            m.addConstr(p_export[k] >= 0.0)

    du_cost = 0.0
    for k in range(H):
        if k == 0:
            du_cost += gp.quicksum((u[k, j] - u_prev_vec[j]) * (u[k, j] - u_prev_vec[j]) for j in range(ctrl_count))
        else:
            du_cost += gp.quicksum((u[k, j] - u[k-1, j]) * (u[k, j] - u[k-1, j]) for j in range(ctrl_count))

    obj = 0.0
    terminal_weight = float(getattr(cfg, "w_terminal", 1.0))
    pmv_target = float(getattr(cfg, "pmv_target", -0.5))
    for k in range(H + 1):
        stage_w = terminal_weight if k == H else 1.0
        for j in range(nx):
            pmv_ac_err = pmv_ac[k, j] - pmv_target
            pmv_nv_err = pmv_nv[k, j] - pmv_target
            obj += stage_w * (pmv_ac_err * pmv_ac_err + pmv_nv_err * pmv_nv_err)
    obj += cfg.w_du * du_cost
    obj += cfg.w_switch * gp.quicksum(sw[k] for k in range(H))
    m.addConstr(gp.quicksum(sw[k] for k in range(H)) <= 2.0)

    energy_cost = 0.0
    for k in range(H):
        energy_fcu = fcu_energy_coeff * gp.quicksum(delta_cool_fcu[k, j] for j in range(nx))
        energy_pfc = local_energy_coeff * gp.quicksum(delta_cool_pfc[k, p] for p in range(len(pfc_indices)))
        if linearize_energy:
            m.addGenConstrIndicator(z_mode[k], True, energy_fcu_used[k] - energy_fcu == 0.0)
            m.addGenConstrIndicator(z_mode[k], False, energy_fcu_used[k] == 0.0)
            m.addGenConstrIndicator(z_mode[k], False, energy_pfc_used[k] - energy_pfc == 0.0)
            m.addGenConstrIndicator(z_mode[k], True, energy_pfc_used[k] == 0.0)
            energy_cost += energy_fcu_used[k] + energy_pfc_used[k]
        else:
            # energy_cost += z_mode * energy_fcu + (1.0 - z_mode) * energy_pfc
            energy_cost += z_mode[k] * energy_fcu + (1.0 - z_mode[k]) * energy_pfc
    obj += w_energy * energy_cost / energy_norm_kwh

    if cfg.use_pv_objective:
        for k in range(H):
            obj += cfg.w_pv_export * p_export[k]

    if soften_x_bounds:
        obj += w_x_slack * gp.quicksum(s_x_low[k, j] + s_x_high[k, j] for k in range(H + 1) for j in range(nx))

    m.setObjective(obj, GRB.MINIMIZE)
    solver_diagnostics = optimize_miqp_with_numeric_retry(
        m, GRB, verbose=verbose
    )


    u0_vec = np.array([float(u[0, j].X) for j in range(ctrl_count)])
    z0_val = int(round(z_mode[0].X))
    pfc_on0_val = int(round(pfc_on[0].X))

    debug = None
    if verbose or collect_diagnostics:
        comfort_term = 0.0
        for k in range(H + 1):
            stage_w = terminal_weight if k == H else 1.0
            for j in range(nx):
                comfort_term += stage_w * ((float(pmv_ac[k, j].X) - pmv_target) ** 2 + (float(pmv_nv[k, j].X) - pmv_target) ** 2)

        du_term = cfg.w_du * float(sum((float(u[k, j].X) - (float(u_prev_vec[j]) if k == 0 else float(u[k-1, j].X))) ** 2 for k in range(H) for j in range(ctrl_count)))
        switch_term = cfg.w_switch * float(sum(float(sw[k].X) for k in range(H)))
        energy_raw_fcu = [fcu_energy_coeff * float(sum(float(delta_cool_fcu[k, j].X) for j in range(nx))) for k in range(H)]
        energy_raw_pfc = [local_energy_coeff * float(sum(float(delta_cool_pfc[k, p].X) for p in range(len(pfc_indices)))) for k in range(H)]

        if linearize_energy:
            energy_cost_sum = float(sum(float(energy_fcu_used[k].X) + float(energy_pfc_used[k].X) for k in range(H)))
            energy_used0 = float(energy_fcu_used[0].X) + float(energy_pfc_used[0].X)
        else:
            energy_cost_sum = float(sum(float(z_mode[k].X) * energy_raw_fcu[k] + (1.0 - float(z_mode[k].X)) * energy_raw_pfc[k] for k in range(H)))
            energy_used0 = float(z_mode[0].X) * energy_raw_fcu[0] + (1.0 - float(z_mode[0].X)) * energy_raw_pfc[0]
        energy_term = w_energy * energy_cost_sum / energy_norm_kwh

        pv_term = 0.0
        if cfg.use_pv_objective:
            for k in range(H):
                pv_term += cfg.w_pv_export * float(p_export[k].X)

        x_slack_term = 0.0
        if soften_x_bounds:
            x_slack_term = w_x_slack * float(sum(float(s_x_low[k, j].X) + float(s_x_high[k, j].X) for k in range(H + 1) for j in range(nx)))

        x_next_mean_ac0 = None
        x_next_mean_nv0 = None
        if A0_ac is not None and A0_nv is not None:
            try:
                x_next_ac0 = A0_ac @ x0_vec + B0_ac @ u0_vec + g0_ac
                u0_pfc = np.asarray([u0_vec[pfc_indices[0]], u0_vec[pfc_indices[1]]], dtype=float)
                x_next_nv0 = A0_nv @ x0_vec + B0_nv @ u0_pfc + g0_nv
                x_next_mean_ac0 = float(np.mean(x_next_ac0))
                x_next_mean_nv0 = float(np.mean(x_next_nv0))
            except Exception:
                x_next_mean_ac0 = None
                x_next_mean_nv0 = None

        solver_mip_gap_value = float("nan")
        solver_mip_gap_available = 0
        try:
            mip_gap_candidate = float(m.MIPGap)
        except Exception:
            mip_gap_candidate = float("nan")
        if np.isfinite(mip_gap_candidate) and mip_gap_candidate >= 0.0:
            solver_mip_gap_value = mip_gap_candidate
            solver_mip_gap_available = 1

        debug = {
            "predicted_max_temp_c": float(
                max(x[k, j].X for k in range(H + 1) for j in range(nx))
            ),
            "predicted_upper_slack_sum": float(
                sum(s_x_high[k, j].X for k in range(H + 1) for j in range(nx))
            ),
            "predicted_upper_slack_positive": int(
                any(
                    s_x_high[k, j].X > 1e-8
                    for k in range(H + 1)
                    for j in range(nx)
                )
            ),
            "solver_initial_status": int(solver_diagnostics.initial_status),
            "solver_final_status": int(solver_diagnostics.final_status),
            "solver_status": int(solver_diagnostics.final_status),
            "numeric_retry_used": int(solver_diagnostics.numeric_retry_used),
            "solver_retry_count": int(solver_diagnostics.retry_count),
            "inf_or_unbd_disambiguation_used": int(
                solver_diagnostics.inf_or_unbd_disambiguation_used
            ),
            "solver_inf_or_unbd_disambiguation_count": int(
                solver_diagnostics.inf_or_unbd_disambiguation_count
            ),
            "solver_initial_runtime_s": float(
                solver_diagnostics.initial_runtime_s
            ),
            "solver_retry_runtime_s": float(
                solver_diagnostics.retry_runtime_s
            ),
            "solver_inf_or_unbd_disambiguation_runtime_s": float(
                solver_diagnostics.inf_or_unbd_disambiguation_runtime_s
            ),
            "solver_total_runtime_s": float(
                solver_diagnostics.total_runtime_s
            ),
            "solver_retry_numeric_focus": (
                float(solver_diagnostics.retry_numeric_focus)
                if solver_diagnostics.retry_numeric_focus is not None
                else float("nan")
            ),
            "solver_runtime_s": float(solver_diagnostics.total_runtime_s),
            "solver_mip_gap": solver_mip_gap_value,
            "solver_mip_gap_available": solver_mip_gap_available,
            "obj": float(m.ObjVal),
            "z0": z0_val,
            "rain_force_ac_now": bool(rain_force_ac_now),
            "breakdown": {
                "comfort": float(comfort_term),
                "du": float(du_term),
                "switch": float(switch_term),
                "energy": float(energy_term),
                "x_slack": float(x_slack_term),
                "pv": float(pv_term),
            },
            "k0": {
                "T_out": float(t_out_fore[0]),
                "sw0": float(sw[0].X),
                "rh_nv": float(rh_nv_fore_h[0]),
                "pmv_target": float(pmv_target),
                "pmv_ac_abs_sum": float(sum(abs(float(pmv_ac[0, j].X)) for j in range(nx))),
                "pmv_nv_abs_sum": float(sum(abs(float(pmv_nv[0, j].X)) for j in range(nx))),
                "pmv_ac_sq_sum": float(sum(float(pmv_ac[0, j].X) ** 2 for j in range(nx))),
                "pmv_nv_sq_sum": float(sum(float(pmv_nv[0, j].X) ** 2 for j in range(nx))),
                "pmv_ac_dev_sq_sum": float(sum((float(pmv_ac[0, j].X) - pmv_target) ** 2 for j in range(nx))),
                "pmv_nv_dev_sq_sum": float(sum((float(pmv_nv[0, j].X) - pmv_target) ** 2 for j in range(nx))),
                "x_high_sum": float(sum(float(s_x_high[0, j].X) for j in range(nx))),
                "energy_fcu0_kwh": float(energy_raw_fcu[0]),
                "energy_pfc0_kwh": float(energy_raw_pfc[0]),
                "energy_used0_kwh": float(energy_used0),
                "x_next_mean_ac0": x_next_mean_ac0,
                "x_next_mean_nv0": x_next_mean_nv0,
            }
        }

    return {"u0": u0_vec, "z0": z0_val, "pfc_on0": pfc_on0_val, "obj": float(m.ObjVal), "debug": debug}


def simulate_day(
    day_idx: int,
    hooks: RepoHooks,
    cfg: MPCConfig,
    start_hhmm: str = "07:30",
    end_hhmm: str = "19:00",
    init_u: float = 24.0,
    init_z: int = 1,
    init_state_cols: Optional[List[str]] = None,
    state_labels: Optional[List[str]] = None,
    progress_fn: Optional[Callable[[int, int, pd.Timestamp], None]] = None,
    verbose: bool = False,
    forecast_source: str = "observed",
    lstm64_future: Optional[pd.DataFrame] = None,
    forecast_case_name: str = "MIQP",
    forecast_controller_objective: str = "not_applicable",
    forecast_audit_records: Optional[List[pd.DataFrame]] = None,
    decision_diagnostic_records: Optional[list[dict]] = None,
) -> pd.DataFrame:
    df_1min = hooks.load_day_dataframe(day_idx).copy().sort_index()

    date0 = df_1min.index[0].date()
    t_start = pd.Timestamp(f"{date0} {start_hhmm}")
    t_end = pd.Timestamp(f"{date0} {end_hhmm}")
    df_sim = df_1min.loc[t_start:t_end - pd.Timedelta(minutes=1)]
    if df_sim.empty:
        raise ValueError("No data in the requested window.")

    if init_state_cols is None:
        init_state_cols = getattr(hooks, 'state_cols', None) or [df_sim.columns[0]]
    if isinstance(init_state_cols, str):
        init_state_cols = [init_state_cols]

    x = df_sim.iloc[0][init_state_cols].astype(float).to_numpy(copy=True)
    u_prev = np.asarray(init_u, dtype=float).reshape(-1)
    if u_prev.size == 1:
        u_prev = np.full(len(AC_CONTROL_IDX), float(u_prev.item()))
    if u_prev.size != len(AC_CONTROL_IDX):
        raise ValueError(f"init_u must be scalar or length {len(AC_CONTROL_IDX)} vector")
    z_prev = int(init_z)

    ctrl_step = cfg.ctrl_period_min
    H = cfg.horizon_min // cfg.ctrl_period_min
    times = df_sim.index
    rows = []

    labels = state_labels or getattr(hooks, 'state_cols', None)
    if labels is None:
        labels = [f"x{j}" for j in range(len(x))]

    ctrl_times = list(times[::ctrl_step])
    total_steps = len(ctrl_times)
    lock_steps = 0
    rain_lock_steps = 0

    for step_idx, t in enumerate(ctrl_times):
        if progress_fn is not None:
            progress_fn(step_idx, total_steps, t)
        else:
            print(f"[simulate_day] step {step_idx + 1}/{total_steps} at {t}")

        print(f"[simulate_day] mean zone temp (pre-solve): {float(np.mean(x)):.3f}")

        t_fore_end = t + pd.Timedelta(minutes=cfg.horizon_min)
        df_fore_1min = df_1min.loc[t:t_fore_end - pd.Timedelta(minutes=1)]
        if len(df_fore_1min) < cfg.horizon_min:
            df_fore_1min = df_fore_1min.reindex(
                pd.date_range(t, periods=cfg.horizon_min, freq="1min"),
                method="nearest"
            )

        df_fore_5min = df_fore_1min.resample(f"{cfg.ctrl_period_min}min").mean().iloc[:H]
        future_forecast = apply_future_data_source(
            df_fore_5min,
            forecast_source,
            decision_time=t,
            lstm64_future=lstm64_future,
        )
        df_fore_5min = future_forecast.frame
        current_exog_row = df_1min.loc[t]
        rain_lock_steps_before = rain_lock_steps
        rain_force_ac_now = observed_rain_forces_ac(cfg, current_exog_row, rain_lock_steps_before)
        rain_lock_steps_after = update_observed_rain_lock_steps(cfg, current_exog_row, rain_lock_steps_before)
        df_fore_5min_mpc = neutralize_rain_for_mpc_prediction(cfg, df_fore_5min)
        if forecast_audit_records is not None:
            forecast_audit_records.append(
                build_forecast_audit(
                    df_fore_5min_mpc,
                    future_forecast.boundary_completed,
                    source=forecast_source,
                    case_name=forecast_case_name,
                    decision_time=t,
                    ctrl_period_min=cfg.ctrl_period_min,
                    controller_objective=forecast_controller_objective,
                    provenance=future_forecast.provenance,
                    base_forecast=future_forecast.frame,
                )
            )
        pre_lock_steps = lock_steps
        effective_lock_steps = 0 if rain_force_ac_now else lock_steps
        if verbose and rain_force_ac_now:
            print(f"[MPC-Explain] observed rain/lockout active; forcing current move to AC (rain lock steps before solve: {rain_lock_steps_before}).")
        elif verbose and pre_lock_steps > 0:
            print(f"[MPC-Explain] dwell lock active for this step (remaining lock steps before solve: {pre_lock_steps}); mode is constrained by dwell policy.")

        try:
            sol = solve_miqp(cfg, x0=x, u_prev=u_prev, z_prev=z_prev, t0=t, exog_5min_forecast=df_fore_5min_mpc, hooks=hooks, lock_steps=effective_lock_steps, verbose=verbose, rain_force_ac_now=rain_force_ac_now,
                collect_diagnostics=decision_diagnostic_records is not None)
        except RuntimeError as exc:
            raise RuntimeError(
                f"{exc}; step={step_idx + 1}/{total_steps}, t={t}, z_prev={z_prev}, lock_steps={effective_lock_steps}, rain_force_ac_now={rain_force_ac_now}, T_mean={float(np.mean(x)):.3f}"
            ) from exc
        u0_vec, z0 = sol['u0'], sol['z0']

        if verbose and pre_lock_steps == 0 and not rain_force_ac_now:
            # Debug-only true counterfactual: force z0 and re-solve from the same state/forecast.
            try:
                sol_cf_ac = solve_miqp(cfg, x0=x, u_prev=u_prev, z_prev=z_prev, t0=t, exog_5min_forecast=df_fore_5min_mpc, hooks=hooks, lock_steps=0, verbose=True, force_z0=1)
                sol_cf_nv = solve_miqp(cfg, x0=x, u_prev=u_prev, z_prev=z_prev, t0=t, exog_5min_forecast=df_fore_5min_mpc, hooks=hooks, lock_steps=0, verbose=True, force_z0=0)
                dbg = sol.get('debug') or {}
                dbg_ac = sol_cf_ac.get('debug') or {}
                dbg_nv = sol_cf_nv.get('debug') or {}
                dbg['counterfactual_z0'] = {
                    'ac': {
                        'obj': float(sol_cf_ac.get('obj', np.nan)),
                        'x1_mean': float((dbg_ac.get('k0') or {}).get('x_next_mean_ac0', np.nan)),
                        'breakdown': dict(dbg_ac.get('breakdown') or {}),
                    },
                    'nv': {
                        'obj': float(sol_cf_nv.get('obj', np.nan)),
                        'x1_mean': float((dbg_nv.get('k0') or {}).get('x_next_mean_nv0', np.nan)),
                        'breakdown': dict(dbg_nv.get('breakdown') or {}),
                    },
                }
                sol['debug'] = dbg
            except Exception as exc:
                dbg = sol.get('debug') or {}
                dbg['counterfactual_z0_error'] = str(exc)
                sol['debug'] = dbg

            _explain_mpc_decision(step_idx, total_steps, t, sol.get('debug'))
        pfc_on0 = sol.get('pfc_on0', 0)
        if decision_diagnostic_records is not None:
            debug = sol.get('debug') or {}
            breakdown = debug.get('breakdown') or {}
            decision_diagnostic_records.append({
                'date': t.date().isoformat(),
                'case': forecast_case_name,
                'decision_time': t.isoformat(),
                'controller_objective': forecast_controller_objective,
                'forecast_source': forecast_source,
                'w_x_slack': float(getattr(cfg, 'w_x_slack')),
                'chosen_mode_z': int(z0),
                'previous_mode_z': int(z_prev),
                'pfcu_on': int(pfc_on0),
                'dwell_lock_steps_before': int(pre_lock_steps),
                'effective_dwell_lock_steps': int(effective_lock_steps),
                'rain_observed_active': int(
                    current_observed_rain_active(cfg, current_exog_row)
                ),
                'rain_force_ac': int(rain_force_ac_now),
                'rain_lock_steps_before': int(rain_lock_steps_before),
                'rain_lock_steps_after': int(rain_lock_steps_after),
                'objective_total': float(sol.get('obj', np.nan)),
                'objective_comfort': float(breakdown.get('comfort', np.nan)),
                'objective_du': float(breakdown.get('du', np.nan)),
                'objective_switch': float(breakdown.get('switch', np.nan)),
                'objective_energy': float(breakdown.get('energy', np.nan)),
                'objective_x_slack': float(breakdown.get('x_slack', np.nan)),
                'objective_pv': float(breakdown.get('pv', np.nan)),
                'predicted_max_temp_c': float(
                    debug.get('predicted_max_temp_c', np.nan)
                ),
                'predicted_upper_slack_sum': float(
                    debug.get('predicted_upper_slack_sum', np.nan)
                ),
                'predicted_upper_slack_positive': int(
                    debug.get('predicted_upper_slack_positive', 0)
                ),
                'solver_status': int(debug.get('solver_status', -1)),
                'solver_initial_status': int(
                    debug.get('solver_initial_status', -1)
                ),
                'solver_final_status': int(
                    debug.get('solver_final_status', -1)
                ),
                'numeric_retry_used': int(
                    debug.get('numeric_retry_used', 0)
                ),
                'solver_retry_count': int(
                    debug.get('solver_retry_count', 0)
                ),
                'inf_or_unbd_disambiguation_used': int(
                    debug.get('inf_or_unbd_disambiguation_used', 0)
                ),
                'solver_inf_or_unbd_disambiguation_count': int(
                    debug.get(
                        'solver_inf_or_unbd_disambiguation_count', 0
                    )
                ),
                'solver_initial_runtime_s': float(
                    debug.get('solver_initial_runtime_s', np.nan)
                ),
                'solver_retry_runtime_s': float(
                    debug.get('solver_retry_runtime_s', np.nan)
                ),
                'solver_inf_or_unbd_disambiguation_runtime_s': float(
                    debug.get(
                        'solver_inf_or_unbd_disambiguation_runtime_s', np.nan
                    )
                ),
                'solver_total_runtime_s': float(
                    debug.get('solver_total_runtime_s', np.nan)
                ),
                'solver_retry_numeric_focus': float(
                    debug.get('solver_retry_numeric_focus', np.nan)
                ),
                'solver_runtime_s': float(
                    debug.get('solver_runtime_s', np.nan)
                ),
                'solver_mip_gap': float(
                    debug.get('solver_mip_gap', np.nan)
                ),
                'solver_mip_gap_available': int(
                    debug.get('solver_mip_gap_available', 0)
                ),
            })
        min_dwell = int(getattr(cfg, "min_dwell_steps", 0))
        if min_dwell > 0:
            if rain_force_ac_now:
                lock_steps = 0
            elif z0 != z_prev:
                lock_steps = max(min_dwell - 1, 0)
            else:
                lock_steps = max(lock_steps - 1, 0)
        rain_lock_steps = rain_lock_steps_after

        for i in range(cfg.ctrl_period_min):
            exog_row = df_1min.loc[t + pd.Timedelta(minutes=i)]
            x_next = hooks.plant_step_1min(mode=z0, x=x, u=u0_vec, exog_row=exog_row)
            rec = {"ts": t + pd.Timedelta(minutes=i), "forecast_source": forecast_source, "z": z0, "pfc_on": int(pfc_on0), "rain_status": float(exog_row.get('rain_status', 0.0)), "rain_force_ac": int(rain_force_ac_now), "rain_lock_steps_before": int(rain_lock_steps_before), "rain_lock_steps_after": int(rain_lock_steps_after), "T_out": float(exog_row.get('T_out', exog_row.get('OutdoorTemperatureWindow', np.nan))), "I_solar": float(exog_row.get('I_solar', exog_row.get('Solar Radiation', 0.0)))}
            # per-unit supply temps for plotting
            if z0 == 1:
                rec['fcu01_supply'], rec['fcu02_supply'], rec['fcu03_supply'], rec['fcu04_supply'], rec['fcu05_supply'], rec['pfc01_supply'], rec['pfc02_supply'] = [float(v) for v in u0_vec]
            else:
                rec['fcu01_supply'] = rec['fcu02_supply'] = rec['fcu03_supply'] = rec['fcu04_supply'] = rec['fcu05_supply'] = np.nan
                rec['pfc01_supply'] = float(u0_vec[5])
                rec['pfc02_supply'] = float(u0_vec[6])
            for val, name in zip(x, labels):
                rec[name] = float(val)
            rec['T_mean'] = float(np.mean(x))
            rows.append(rec)
            x = np.asarray(x_next, dtype=float)

        u_prev, z_prev = u0_vec, z0

    out = pd.DataFrame(rows).set_index('ts')
    return out


In [ ]:
# Repo-specific hooks wired to existing surrogates and data loader
class MMVRepoHooks(RepoHooks):
    def __init__(self, data_path: str, ac_model_path: Path, nv_model_path: Path, control_mask_ac=None, control_mask_nv=None, steps_per_ctrl: int = 1, lin_cache_size: int = 512, cache_round_decimals: int = 4, segment_split: str = 'test', test_date_policy: str = 'intersection', min_seg_len: int = 600):
        self.data_path = data_path
        self.device = torch.device('cpu')
        self.ac_model, self.ac_scaler_X, self.ac_scaler_U = load_surrogate(str(ac_model_path), self.device)
        self.nv_model, self.nv_scaler_X, self.nv_scaler_U = load_surrogate(str(nv_model_path), self.device)
        self.state_cols = AC_STATE_COLS
        self.steps_per_ctrl = steps_per_ctrl
        self.segment_split = segment_split
        self.test_date_policy = test_date_policy
        self.min_seg_len = min_seg_len
        self.control_mask_ac = control_mask_ac if control_mask_ac is not None else AC_CONTROL_IDX
        self.control_mask_nv = control_mask_nv if control_mask_nv is not None else NV_CONTROL_IDX
        self._segments = None
        self._lin_cache = OrderedDict()
        self.lin_cache_size = lin_cache_size
        self.cache_round_decimals = cache_round_decimals
        self.lin_cache_hits = 0
        self.lin_cache_misses = 0

    def _segments_cached(self):
        if self._segments is None:
            self._segments = prepare_segments(self.data_path, max_segments=50, split=self.segment_split, test_policy=self.test_date_policy, min_len=self.min_seg_len)
        return self._segments

    def load_day_dataframe(self, day_idx: int = 0) -> pd.DataFrame:
        segments = self._segments_cached()
        if not segments:
            raise ValueError('No clean segments found in data.')

        if not (0 <= int(day_idx) < len(segments)):
            raise ValueError(f'day_idx must be in [0, {len(segments)-1}], got {day_idx}')
        target = segments[int(day_idx)]

        df = target.copy().set_index('date')
        df['T_out'] = df['OutdoorTemperatureWindow']
        df['I_solar'] = df['Solar Radiation']
        return df

    def _select_model(self, mode: str):
        if mode == 'ac':
            return self.ac_model, self.ac_scaler_X, self.ac_scaler_U
        else:
            return self.nv_model, self.nv_scaler_X, self.nv_scaler_U

    def _control_mask(self, mode: str):
        return self.control_mask_ac if mode == "ac" else self.control_mask_nv

    def _linearization_cache_key(self, mode_name: str, x_oper, u_oper, exog_fore_5min: pd.Series):
        x_key = tuple(np.round(np.asarray(x_oper, dtype=float).reshape(-1), self.cache_round_decimals))
        if mode_name == 'ac':
            u_vec = np.asarray(u_oper, dtype=float).reshape(-1)
            if u_vec.size == 1:
                u_vec = np.repeat(u_vec, len(AC_CONTROL_IDX))
            u_key = tuple(np.round(u_vec, self.cache_round_decimals))
            exog_cols = AC_INPUT_COLS
        else:
            u_vec = np.asarray(u_oper, dtype=float).reshape(-1)[-2:]
            u_key = tuple(np.round(u_vec, self.cache_round_decimals))
            exog_cols = NV_INPUT_COLS_BASE
        exog_vals = tuple(np.round([float(exog_fore_5min.get(col, 0.0)) for col in exog_cols], self.cache_round_decimals))
        return (mode_name, x_key, u_key, exog_vals)

    def _cache_lookup(self, key):
        cached = self._lin_cache.get(key)
        if cached is None:
            self.lin_cache_misses += 1
            return None
        self.lin_cache_hits += 1
        self._lin_cache.move_to_end(key)
        return tuple(np.array(arr, copy=True) for arr in cached)

    def _cache_store(self, key, A5, B5, c5):
        self._lin_cache[key] = (A5, B5, c5)
        self._lin_cache.move_to_end(key)
        if len(self._lin_cache) > self.lin_cache_size:
            self._lin_cache.popitem(last=False)

    def reset_cache(self):
        self._lin_cache.clear()
        self.lin_cache_hits = 0
        self.lin_cache_misses = 0

    def cache_stats(self):
        return {"hits": self.lin_cache_hits, "misses": self.lin_cache_misses, "entries": len(self._lin_cache)}

    def plant_step_1min(self, mode: int, x, u, exog_row: pd.Series):
        mode_name = 'ac' if mode == 1 else 'nv'
        model, sx, su = self._select_model(mode_name)
        u_vec = build_u_vector(exog_row, mode_name, u if mode_name == "ac" else np.asarray(u)[-2:])
        return forward_surrogate(model, sx, su, x, u_vec, self.device)

    def linearize_5min(self, mode: int, x_oper, u_oper, exog_fore_5min: pd.Series):
        mode_name = 'ac' if mode == 1 else 'nv'
        key = self._linearization_cache_key(mode_name, x_oper, u_oper, exog_fore_5min)
        cached = self._cache_lookup(key)
        if cached is not None:
            return cached

        model, sx, su = self._select_model(mode_name)
        u_vec = build_u_vector(exog_fore_5min, mode_name, u_oper if mode_name == "ac" else np.asarray(u_oper)[-2:])
        lin = linearize_surrogate(model, sx, su, x_oper, u_vec, self.device)
        mask = np.asarray(self._control_mask(mode_name), dtype=int)

        # Reduce full-input linearization to control-only form while preserving
        # the linearization point contribution from non-control (exogenous) inputs.
        omit_mask = np.ones(lin.B.shape[1], dtype=bool)
        omit_mask[mask] = False
        B_sel = lin.B[:, mask]
        c_reduced = lin.c + lin.B[:, omit_mask] @ u_vec[omit_mask]

        A5, B5, c5 = lift_linearization(lin.A, B_sel, c_reduced, steps=self.steps_per_ctrl)
        self._cache_store(key, A5, B5, c5)
        return tuple(np.array(arr, copy=True) for arr in (A5, B5, c5))


# Convenience factory
def make_default_hooks(**kwargs):
    return MMVRepoHooks(**kwargs)


## Fixed configuration and three-weight grid


In [ ]:
segment_split = 'test+val'
test_date_policy = 'union'
max_segments_env = os.environ.get('MMV4EF_MAX_SEGMENTS', '').strip()
max_segments_to_run = int(max_segments_env) if max_segments_env else None
if max_segments_to_run is not None and max_segments_to_run < 1:
    raise ValueError(
        'MMV4EF_MAX_SEGMENTS must be a positive integer when set.'
    )

window_mode = os.environ.get('MMV4EF_WINDOW', 'full').strip().lower()
if window_mode == 'smoke':
    run_window = dict(start_hhmm='07:30', end_hhmm='09:00')
    mpc_overrides = dict(horizon_min=60, mip_time_limit=30)
elif window_mode == 'full':
    run_window = dict(start_hhmm='07:30', end_hhmm='19:00')
    mpc_overrides = dict(horizon_min=60)
else:
    raise ValueError("MMV4EF_WINDOW must be 'full' or 'smoke'.")

save_timeseries = True
show_daily_figures = False

pv_source_area_sqft = 5000.0
building_area_sqft = 75.0
pv_scale_factor = building_area_sqft / pv_source_area_sqft

best_model = pd.read_csv(PV_COEFFICIENT_PATH).iloc[0]
coeffs = {
    'a1': float(best_model['a1']),
    'a2': float(best_model['a2']),
    'a3': float(best_model['a3']),
}

SIMPLE_WEIGHTS = dict(
    energy_weight=1.667e3,
    smoothness_weight=1e-1,
    pv_self_consumption_weight=20.0,
)

FIXED_MPC_KWARGS = dict(
    T_ref_ac=27.0,
    T_nv_max=30.5,
    T_nv_soft=30.0,
    x_min=22.0,
    x_max=30.0,
    pmv_target=-0.5,
    pmv_model='linear',
    rh_ac_fixed=65.0,
    rh_nv_source_col='OutdoorHumidityWindow',
    pmv_big_m=5.0,
    energy_norm_kwh=10.0,
    cop_fcu=5.0,
    cop_pfc=5.0,
    T_out_nv_ref=30.0,
    min_dwell_steps=12,
    enforce_current_rain_safety=True,
    rain_active_threshold=0.5,
    rain_lockout_min=30,
    ignore_rain_in_mpc_prediction=True,
    linearize_energy=True,
    w_terminal=2e1,
    w_switch=1e1,
    w_x_slack=SWEEP_WEIGHTS[0],
)

FCU_SUPPLY_COLS = [
    'fcu01_supply', 'fcu02_supply', 'fcu03_supply', 'fcu04_supply',
    'fcu05_supply',
]
PFC_SUPPLY_COLS = ['pfc01_supply', 'pfc02_supply']


def build_simple_pmv_cfg(
    energy_weight: float = 1.667e3,
    smoothness_weight: float = 1e-1,
    pv_self_consumption_weight: float = 20.0,
    use_pv_objective: bool = False,
    **overrides,
) -> MPCConfig:
    cfg_kwargs = dict(FIXED_MPC_KWARGS)
    cfg_kwargs.update(
        w_energy=energy_weight,
        w_du=smoothness_weight,
        use_pv_objective=use_pv_objective,
    )
    cfg_kwargs.update(overrides)
    if use_pv_objective:
        cfg_kwargs.update(
            pv_model='appendix_b',
            pv_a1=coeffs['a1'],
            pv_a2=coeffs['a2'],
            pv_a3=coeffs['a3'],
            pv_source_area_sqft=pv_source_area_sqft,
            building_area_sqft=building_area_sqft,
            pv_scale_factor=pv_scale_factor,
            pv_pmax=None,
            w_pv_export=pv_self_consumption_weight,
        )
    return MPCConfig(**cfg_kwargs)


base_cfg = build_simple_pmv_cfg(
    **SIMPLE_WEIGHTS, use_pv_objective=False, **mpc_overrides
)
pv_cfg = build_simple_pmv_cfg(
    **SIMPLE_WEIGHTS, use_pv_objective=True, **mpc_overrides
)
pv_metric_cfg = build_simple_pmv_cfg(
    **SIMPLE_WEIGHTS, use_pv_objective=True, **mpc_overrides
)

BASELINE_CASES = ('AC baseline', 'RBC baseline')
MIQP_CASE_SPECS = tuple(
    spec
    for weight in SWEEP_WEIGHTS
    for spec in (
        (
            f'MIQP no PV | observed future | wx={int(weight)}',
            replace(base_cfg, w_x_slack=weight),
            'no_pv',
            'observed',
            weight,
        ),
        (
            f'MIQP onsite PV | observed future | wx={int(weight)}',
            replace(pv_cfg, w_x_slack=weight),
            'onsite_pv',
            'observed',
            weight,
        ),
    )
)
CASE_ORDER = [*BASELINE_CASES, *(spec[0] for spec in MIQP_CASE_SPECS)]
CASE_METADATA = {
    'AC baseline': {
        'controller_objective': 'baseline_ac',
        'forecast_source': 'not_applicable',
        'w_x_slack': np.nan,
    },
    'RBC baseline': {
        'controller_objective': 'baseline_rbc',
        'forecast_source': 'not_applicable',
        'w_x_slack': np.nan,
    },
    **{
        name: {
            'controller_objective': objective,
            'forecast_source': forecast_source,
            'w_x_slack': float(weight),
        }
        for name, _, objective, forecast_source, weight in MIQP_CASE_SPECS
    },
}

hooks = make_default_hooks(
    data_path=str(L14_DATA_PATH),
    ac_model_path=AC_MODEL_PATH,
    nv_model_path=NV_MODEL_PATH,
    test_date_policy=test_date_policy,
    segment_split=segment_split,
    min_seg_len=600,
)
segments = hooks._segments_cached()
if not segments:
    raise ValueError('No clean segments are available for the requested split/policy.')

if max_segments_to_run is None:
    selected_indices = list(range(len(segments)))
else:
    selected_indices = list(range(min(max_segments_to_run, len(segments))))
selected_dates = [
    segments[index]['date'].iloc[0].date().isoformat()
    for index in selected_indices
]
run_suffix = 'all' if max_segments_to_run is None else f'first{len(selected_indices)}'
STUDY_SLUG = 'control_case_comparison_w_x_slack_sweep'
# The run directory already carries the collision-proof run ID. Keep artifact
# names short enough for the nested Windows worktree path.
run_stem = 'wx_sweep'

print({
    'output_dir': str(OUTPUT_DIR.relative_to(REPO_ROOT)),
    'selected_dates': selected_dates,
    'window_mode': window_mode,
    'run_window': run_window,
    'horizon_min': base_cfg.horizon_min,
    'sweep_weights': SWEEP_WEIGHTS,
    'case_count': len(CASE_ORDER),
    'study_slug': STUDY_SLUG,
    'gurobi_available': GUROBI_OK,
})


## Fixed-policy baselines


In [ ]:
# Rule-based controller parameters from rbc.md
DEADBAND = 0.3
NV_CC_BASE = 29.0
CC_AC_BASE = 30.0
NV_CC_HIGH = NV_CC_BASE + DEADBAND
NV_CC_LOW = NV_CC_BASE - DEADBAND
CC_AC_HIGH = CC_AC_BASE + DEADBAND
CC_AC_LOW = CC_AC_BASE - DEADBAND
MIN_MODE_LOCK = 5
RAIN_LOCKOUT = 30
AC_ZONE_SETPOINT_C = 27.0
AC_THERMOSTAT_GAIN = 3.0


@dataclass
class ControllerState:
    mode: str
    last_switch_idx: Optional[int] = None
    rain_hold: int = 0


def pick_initial_rbc_mode(row: pd.Series) -> str:
    oa = float(row['OutdoorTemperatureWindow'])
    rain = float(row['rain_status'])
    if rain >= 1:
        return 'ac'
    if oa >= CC_AC_HIGH:
        return 'ac'
    if oa >= NV_CC_HIGH:
        return 'cc'
    return 'nv'


def update_rbc_mode(row: pd.Series, idx: int, state: ControllerState) -> str:
    oa = float(row['OutdoorTemperatureWindow'])
    rain = float(row['rain_status'])

    if rain >= 1:
        state.rain_hold = RAIN_LOCKOUT
        if state.mode != 'ac':
            state.mode = 'ac'
            state.last_switch_idx = idx
        return state.mode

    if state.rain_hold > 0:
        state.rain_hold -= 1
        if state.mode != 'ac':
            state.mode = 'ac'
            state.last_switch_idx = idx
        return state.mode

    locked = state.last_switch_idx is not None and (idx - state.last_switch_idx) < MIN_MODE_LOCK
    if locked:
        return state.mode

    prev_mode = state.mode
    if state.mode == 'nv':
        if oa >= NV_CC_HIGH:
            state.mode = 'cc'
    elif state.mode == 'cc':
        if oa >= CC_AC_HIGH:
            state.mode = 'ac'
        elif oa <= NV_CC_LOW:
            state.mode = 'nv'
    elif state.mode == 'ac':
        if oa <= CC_AC_LOW:
            state.mode = 'cc'
    else:
        state.mode = 'nv'

    if state.mode != prev_mode:
        state.last_switch_idx = idx
    return state.mode


def ac_supply_for_setpoint(x, cfg: MPCConfig, setpoint_c: float = AC_ZONE_SETPOINT_C) -> np.ndarray:
    mean_temp = float(np.mean(np.asarray(x, dtype=float)))
    cooling_error = max(mean_temp - float(setpoint_c), 0.0)
    fcu_supply = np.clip(cfg.u_max_ac - AC_THERMOSTAT_GAIN * cooling_error, cfg.u_min_ac, cfg.u_max_ac)
    return np.array([fcu_supply] * 5 + [cfg.u_min_pfc, cfg.u_min_pfc], dtype=float)


def control_vector_for_baseline(mode_label: str, x, exog_row: pd.Series, cfg: MPCConfig) -> tuple[int, np.ndarray, int]:
    if mode_label == 'ac':
        return 1, ac_supply_for_setpoint(x, cfg), 0

    u_vec = np.array([np.nan] * 5 + [cfg.u_min_pfc, cfg.u_min_pfc], dtype=float)
    if mode_label == 'nv':
        oa = float(exog_row['OutdoorTemperatureWindow'])
        u_vec[-2:] = [oa, oa]
        return 0, u_vec, 0

    if mode_label == 'cc':
        u_vec[-2:] = [cfg.u_min_pfc, cfg.u_min_pfc]
        return 0, u_vec, 1

    raise ValueError(f'Unknown mode_label={mode_label!r}')


def simulate_policy_day(
    case_name: str,
    hooks: RepoHooks,
    cfg: MPCConfig,
    day_idx: int,
    policy: str,
    start_hhmm: str = '07:30',
    end_hhmm: str = '19:00',
) -> pd.DataFrame:
    df_1min = hooks.load_day_dataframe(day_idx).copy().sort_index()
    date0 = df_1min.index[0].date()
    t_start = pd.Timestamp(f'{date0} {start_hhmm}')
    t_end = pd.Timestamp(f'{date0} {end_hhmm}')
    df_sim = df_1min.loc[t_start:t_end - pd.Timedelta(minutes=1)]
    if df_sim.empty:
        raise ValueError('No data in the requested baseline window.')

    x = df_sim.iloc[0][hooks.state_cols].astype(float).to_numpy(copy=True)
    rbc_state = None
    rows = []

    for i, (ts, exog_row) in enumerate(df_sim.iterrows()):
        if policy == 'ac':
            mode_label = 'ac'
        elif policy == 'rbc':
            if rbc_state is None:
                rbc_state = ControllerState(mode=pick_initial_rbc_mode(exog_row))
            mode_label = update_rbc_mode(exog_row, i, rbc_state)
        else:
            raise ValueError(f'Unknown baseline policy={policy!r}')

        z_mode, u_vec, pfc_on = control_vector_for_baseline(mode_label, x, exog_row, cfg)
        rec = {
            'ts': ts,
            'case': case_name,
            'z': z_mode,
            'mode_label': mode_label.upper(),
            'pfc_on': int(pfc_on),
            'T_out': float(exog_row.get('T_out', exog_row.get('OutdoorTemperatureWindow', np.nan))),
            'I_solar': float(exog_row.get('I_solar', exog_row.get('Solar Radiation', 0.0))),
            'fcu01_supply': float(u_vec[0]) if z_mode == 1 else np.nan,
            'fcu02_supply': float(u_vec[1]) if z_mode == 1 else np.nan,
            'fcu03_supply': float(u_vec[2]) if z_mode == 1 else np.nan,
            'fcu04_supply': float(u_vec[3]) if z_mode == 1 else np.nan,
            'fcu05_supply': float(u_vec[4]) if z_mode == 1 else np.nan,
            'pfc01_supply': float(u_vec[5]),
            'pfc02_supply': float(u_vec[6]),
        }
        for val, name in zip(x, hooks.state_cols):
            rec[name] = float(val)
        rec['T_mean'] = float(np.mean(x))
        rows.append(rec)

        x = np.asarray(hooks.plant_step_1min(mode=z_mode, x=x, u=u_vec, exog_row=exog_row), dtype=float)

    return pd.DataFrame(rows).set_index('ts')


## Metrics, solver provenance, and validation helpers


In [ ]:
def enrich_results(
    sim_df: pd.DataFrame,
    cfg: MPCConfig,
    case_name: Optional[str] = None,
    segment_date: Optional[str] = None,
    forecast_source: Optional[str] = None,
    controller_objective: Optional[str] = None,
    w_x_slack: Optional[float] = None,
) -> pd.DataFrame:
    out = sim_df.copy()
    metadata = CASE_METADATA.get(case_name, {})
    if case_name is not None:
        out['case'] = case_name
    if segment_date is not None:
        out['date'] = segment_date
    out['forecast_source'] = (
        forecast_source
        if forecast_source is not None
        else metadata.get('forecast_source', 'not_applicable')
    )
    out['controller_objective'] = (
        controller_objective
        if controller_objective is not None
        else metadata.get('controller_objective', 'not_applicable')
    )
    out['w_x_slack'] = (
        float(w_x_slack)
        if w_x_slack is not None
        else metadata.get('w_x_slack', np.nan)
    )
    out['pv_kw'] = [
        pv_available_power_kw(cfg, I_solar=i_solar, T_out=t_out)
        for i_solar, t_out in zip(out['I_solar'], out['T_out'])
    ]
    out['hvac_kw'] = [
        hvac_power_heat_exchange_kw(
            cfg,
            z_mode=row['z'],
            zone_temps=row[AC_STATE_COLS].to_numpy(dtype=float),
            fcu_supplies=row[FCU_SUPPLY_COLS]
            .fillna(row['T_out'])
            .to_numpy(dtype=float),
            pfc_supplies=row[PFC_SUPPLY_COLS].to_numpy(dtype=float),
            T_out=row['T_out'],
        )
        for _, row in out.iterrows()
    ]
    out['grid_kw'] = np.maximum(out['hvac_kw'] - out['pv_kw'], 0.0)
    out['export_kw'] = np.maximum(out['pv_kw'] - out['hvac_kw'], 0.0)
    out['self_kw'] = np.minimum(out['hvac_kw'], out['pv_kw'])
    return out


def peak_window_metrics(df: pd.DataFrame) -> dict:
    timestamps = pd.DatetimeIndex(
        pd.to_datetime(df['ts'] if 'ts' in df.columns else df.index)
    )
    if timestamps.isna().any():
        raise ValueError('Peak-window metrics require finite timestamps.')
    minutes = timestamps.hour * 60 + timestamps.minute
    # Match time_in_windows exactly: 07:30-10:00 and 16:30-19:00.
    morning = (minutes >= 7 * 60 + 30) & (minutes < 10 * 60)
    evening = (minutes >= 16 * 60 + 30) & (minutes < 19 * 60)
    configured_union = np.asarray(
        [time_in_windows(timestamp) for timestamp in timestamps], dtype=bool
    )
    if not np.array_equal(morning | evening, configured_union):
        raise RuntimeError('Peak-window masks drifted from time_in_windows.')

    def summarize(mask, prefix: str) -> dict:
        represented_dates = pd.Index(timestamps.normalize().unique())
        window_timestamps = timestamps[mask].floor('min')
        observed_date_minutes = pd.DataFrame({
            'date': window_timestamps.normalize(),
            'minute': window_timestamps.hour * 60 + window_timestamps.minute,
        }).drop_duplicates()
        observed_by_date = (
            observed_date_minutes.groupby('date').size()
            .reindex(represented_dates, fill_value=0)
        )
        minutes_per_complete_window = 150
        available = bool(
            len(represented_dates) > 0
            and observed_by_date.eq(minutes_per_complete_window).all()
        )
        return {
            f'{prefix}_peak_available': int(available),
            f'{prefix}_peak_observed_minutes': int(observed_by_date.sum()),
            f'{prefix}_peak_expected_minutes': int(
                len(represented_dates) * minutes_per_complete_window
            ),
            f'{prefix}_peak_hvac_kw': (
                float(df.loc[mask, 'hvac_kw'].max())
                if available else np.nan
            ),
            f'{prefix}_peak_grid_kw': (
                float(df.loc[mask, 'grid_kw'].max())
                if available else np.nan
            ),
        }

    return {
        **summarize(morning, 'morning'),
        **summarize(evening, 'evening'),
    }


def summarize_case(
    df: pd.DataFrame,
    pv_window_threshold_kw: Optional[float] = None,
) -> dict:
    step_h = 1.0 / 60.0
    if pv_window_threshold_kw is None:
        pv_window_threshold_kw = max(float(df['pv_kw'].max()) * 0.05, 1e-9)
    pv_window = df['pv_kw'] > pv_window_threshold_kw
    hvac_sum = float(df['hvac_kw'].sum())
    pv_sum = float(df['pv_kw'].sum())
    self_sum = float(df['self_kw'].sum())
    window_open = 1 - df['z'].astype(int)
    thermal = temperature_metrics(df, AC_STATE_COLS, bound_c=30.0)
    post5_thermal = {
        f'post5_{name}': value
        for name, value in temperature_metrics(
            df,
            AC_STATE_COLS,
            bound_c=30.0,
            exclude_first_minutes=5,
        ).items()
    }
    below = np.maximum(22.0 - df[AC_STATE_COLS].to_numpy(dtype=float), 0.0)
    rain_force = pd.to_numeric(
        df.get('rain_force_ac', pd.Series(0, index=df.index)),
        errors='coerce',
    ).fillna(0).astype(int)
    rain_lock = pd.to_numeric(
        df.get('rain_lock_steps_before', pd.Series(0, index=df.index)),
        errors='coerce',
    ).fillna(0).astype(int)
    rain_safety = ((rain_force > 0) | (rain_lock > 0)) & (
        df['z'].astype(int) != 1
    )
    peak_windows = peak_window_metrics(df)
    return {
        'hvac_kwh': hvac_sum * step_h,
        'pv_available_kwh': pv_sum * step_h,
        'grid_kwh': float(df['grid_kw'].sum()) * step_h,
        'export_kwh': float(df['export_kw'].sum()) * step_h,
        'self_consumed_kwh': self_sum * step_h,
        'self_consumption_ratio': self_sum / pv_sum if pv_sum > 0 else np.nan,
        'self_sufficiency_ratio': self_sum / hvac_sum if hvac_sum > 0 else np.nan,
        'pv_window_hvac_kwh': float(df.loc[pv_window, 'hvac_kw'].sum()) * step_h,
        'non_pv_window_hvac_kwh': (
            float(df.loc[~pv_window, 'hvac_kw'].sum()) * step_h
        ),
        'pv_window_hvac_share_pct': (
            100.0 * float(df.loc[pv_window, 'hvac_kw'].sum()) / hvac_sum
            if hvac_sum > 0 else np.nan
        ),
        'peak_hvac_kw': float(df['hvac_kw'].max()),
        'peak_grid_kw': float(df['grid_kw'].max()),
        'mean_temp_c': float(df['T_mean'].mean()),
        'max_temp_c': float(df['T_mean'].max()),
        'min_temp_c': float(df['T_mean'].min()),
        'zone_min_c': float(df[AC_STATE_COLS].to_numpy(dtype=float).min()),
        'zone_minutes_below_22': int(np.any(below > 0.0, axis=1).sum()),
        'max_below_22_c': float(below.max()),
        'ac_fraction': float(df['z'].mean()),
        'pfc_on_fraction': float(df['pfc_on'].astype(int).mean()),
        'window_open_fraction': float(window_open.mean()),
        'window_open_minutes': int(window_open.sum()),
        'switch_count': int(
            (df['z'].astype(int).diff().abs().fillna(0) > 0).sum()
        ),
        'rain_forced_ac_minutes': int(rain_force.sum()),
        'rain_lockout_minutes': int((rain_lock > 0).sum()),
        'rain_safety_violation_minutes': int(rain_safety.sum()),
        **thermal,
        **post5_thermal,
        **peak_windows,
    }


def build_metric_table(case_results: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    max_pv = max(float(df['pv_kw'].max()) for df in case_results.values())
    threshold = max(max_pv * 0.05, 1e-9)
    metrics = pd.DataFrame.from_dict(
        {
            name: summarize_case(df, pv_window_threshold_kw=threshold)
            for name, df in case_results.items()
        },
        orient='index',
    ).loc[CASE_ORDER]
    metrics['controller_objective'] = [
        CASE_METADATA[name]['controller_objective'] for name in metrics.index
    ]
    metrics['forecast_source'] = [
        CASE_METADATA[name]['forecast_source'] for name in metrics.index
    ]
    metrics['w_x_slack'] = [
        CASE_METADATA[name]['w_x_slack'] for name in metrics.index
    ]
    ac_hvac = float(metrics.loc['AC baseline', 'hvac_kwh'])
    rbc_hvac = float(metrics.loc['RBC baseline', 'hvac_kwh'])
    metrics['energy_savings_vs_ac_pct'] = (
        100.0 * (ac_hvac - metrics['hvac_kwh']) / ac_hvac
        if ac_hvac > 0.05 else np.nan
    )
    metrics['energy_shedding_vs_rbc_kwh'] = rbc_hvac - metrics['hvac_kwh']
    metrics['energy_shedding_vs_rbc_pct'] = (
        100.0 * (rbc_hvac - metrics['hvac_kwh']) / rbc_hvac
        if rbc_hvac > 0.05 else np.nan
    )
    metrics['energy_savings_vs_rbc_pct'] = metrics[
        'energy_shedding_vs_rbc_pct'
    ]
    metrics['grid_reduction_vs_miqp_no_pv_kwh'] = np.nan
    metrics['load_shifted_to_pv_window_kwh'] = np.nan
    for weight in SWEEP_WEIGHTS:
        no_pv_name = f'MIQP no PV | observed future | wx={int(weight)}'
        pv_name = f'MIQP onsite PV | observed future | wx={int(weight)}'
        reference = metrics.loc[no_pv_name]
        metrics.loc[
            [no_pv_name, pv_name], 'grid_reduction_vs_miqp_no_pv_kwh'
        ] = reference['grid_kwh'] - metrics.loc[
            [no_pv_name, pv_name], 'grid_kwh'
        ]
        metrics.loc[
            [no_pv_name, pv_name], 'load_shifted_to_pv_window_kwh'
        ] = metrics.loc[
            [no_pv_name, pv_name], 'pv_window_hvac_kwh'
        ] - reference['pv_window_hvac_kwh']
    return metrics


DECISION_SUMMARY_COLUMNS = (
    'decision_count',
    'predicted_upper_slack_sum',
    'predicted_upper_slack_positive_fraction',
    'numeric_retry_count',
    'numeric_retry_fraction',
    'inf_or_unbd_disambiguation_count',
    'inf_or_unbd_disambiguation_fraction',
    'solver_initial_runtime_sum_s',
    'solver_retry_runtime_sum_s',
    'solver_inf_or_unbd_disambiguation_runtime_sum_s',
    'solver_total_runtime_sum_s',
    'solver_runtime_sum_s',
    'solver_runtime_mean_s',
    'solver_runtime_max_s',
    'solver_mip_gap_available_fraction',
    'solver_mip_gap_mean',
    'solver_mip_gap_max',
)


def summarize_decision_diagnostics(frame: pd.DataFrame) -> dict:
    if frame.empty:
        return {
            'decision_count': 0,
            'predicted_upper_slack_sum': np.nan,
            'predicted_upper_slack_positive_fraction': np.nan,
            'numeric_retry_count': 0,
            'numeric_retry_fraction': np.nan,
            'inf_or_unbd_disambiguation_count': 0,
            'inf_or_unbd_disambiguation_fraction': np.nan,
            'solver_initial_runtime_sum_s': np.nan,
            'solver_retry_runtime_sum_s': np.nan,
            'solver_inf_or_unbd_disambiguation_runtime_sum_s': np.nan,
            'solver_total_runtime_sum_s': np.nan,
            'solver_runtime_sum_s': np.nan,
            'solver_runtime_mean_s': np.nan,
            'solver_runtime_max_s': np.nan,
            'solver_mip_gap_available_fraction': np.nan,
            'solver_mip_gap_mean': np.nan,
            'solver_mip_gap_max': np.nan,
        }
    gap_available = pd.to_numeric(
        frame['solver_mip_gap_available'], errors='coerce'
    )
    gap_values = pd.to_numeric(frame['solver_mip_gap'], errors='coerce')
    available_gaps = gap_values.loc[gap_available == 1]
    runtime = pd.to_numeric(frame['solver_runtime_s'], errors='coerce')
    retry_used = pd.to_numeric(frame['numeric_retry_used'], errors='coerce')
    inf_or_unbd_disambiguation_used = pd.to_numeric(
        frame['inf_or_unbd_disambiguation_used'], errors='coerce'
    )
    initial_runtime = pd.to_numeric(
        frame['solver_initial_runtime_s'], errors='coerce'
    )
    retry_runtime = pd.to_numeric(
        frame['solver_retry_runtime_s'], errors='coerce'
    )
    inf_or_unbd_disambiguation_runtime = pd.to_numeric(
        frame['solver_inf_or_unbd_disambiguation_runtime_s'], errors='coerce'
    )
    total_runtime = pd.to_numeric(
        frame['solver_total_runtime_s'], errors='coerce'
    )
    return {
        'decision_count': int(len(frame)),
        'predicted_upper_slack_sum': float(
            pd.to_numeric(
                frame['predicted_upper_slack_sum'], errors='coerce'
            ).sum()
        ),
        'predicted_upper_slack_positive_fraction': float(
            pd.to_numeric(
                frame['predicted_upper_slack_positive'], errors='coerce'
            ).mean()
        ),
        'numeric_retry_count': int(retry_used.sum()),
        'numeric_retry_fraction': float(retry_used.mean()),
        'inf_or_unbd_disambiguation_count': int(
            inf_or_unbd_disambiguation_used.sum()
        ),
        'inf_or_unbd_disambiguation_fraction': float(
            inf_or_unbd_disambiguation_used.mean()
        ),
        'solver_initial_runtime_sum_s': float(initial_runtime.sum()),
        'solver_retry_runtime_sum_s': float(retry_runtime.sum()),
        'solver_inf_or_unbd_disambiguation_runtime_sum_s': float(
            inf_or_unbd_disambiguation_runtime.sum()
        ),
        'solver_total_runtime_sum_s': float(total_runtime.sum()),
        'solver_runtime_sum_s': float(runtime.sum()),
        'solver_runtime_mean_s': float(runtime.mean()),
        'solver_runtime_max_s': float(runtime.max()),
        'solver_mip_gap_available_fraction': float(gap_available.mean()),
        'solver_mip_gap_mean': (
            float(available_gaps.mean()) if len(available_gaps) else np.nan
        ),
        'solver_mip_gap_max': (
            float(available_gaps.max()) if len(available_gaps) else np.nan
        ),
    }


def validate_solver_recovery_provenance(frame: pd.DataFrame, grb) -> dict:
    numeric_retry_used = pd.to_numeric(
        frame['numeric_retry_used'], errors='coerce'
    )
    retry_count = pd.to_numeric(
        frame['solver_retry_count'], errors='coerce'
    )
    disambiguation_used = pd.to_numeric(
        frame['inf_or_unbd_disambiguation_used'], errors='coerce'
    )
    disambiguation_count = pd.to_numeric(
        frame['solver_inf_or_unbd_disambiguation_count'], errors='coerce'
    )
    initial_status = pd.to_numeric(
        frame['solver_initial_status'], errors='coerce'
    )
    final_status = pd.to_numeric(
        frame['solver_final_status'], errors='coerce'
    )
    reported_status = pd.to_numeric(
        frame['solver_status'], errors='coerce'
    )
    retry_numeric_focus = pd.to_numeric(
        frame['solver_retry_numeric_focus'], errors='coerce'
    )
    no_recovery = (numeric_retry_used == 0) & (disambiguation_used == 0)
    provenance_valid = bool(
        numeric_retry_used.isin([0, 1]).all()
        and disambiguation_used.isin([0, 1]).all()
        and bool((retry_count == numeric_retry_used).all())
        and bool((disambiguation_count == disambiguation_used).all())
        and bool((numeric_retry_used + disambiguation_used <= 1).all())
        and bool((reported_status == final_status).all())
        and bool((
            initial_status.loc[numeric_retry_used == 1] == grb.NUMERIC
        ).all())
        and bool((
            initial_status.loc[disambiguation_used == 1] == grb.INF_OR_UNBD
        ).all())
        and bool((
            initial_status.loc[no_recovery] == final_status.loc[no_recovery]
        ).all())
        and bool((retry_numeric_focus.loc[numeric_retry_used == 1] == 3).all())
        and retry_numeric_focus.loc[numeric_retry_used == 0].isna().all()
    )

    initial_runtime = pd.to_numeric(
        frame['solver_initial_runtime_s'], errors='coerce'
    )
    retry_runtime = pd.to_numeric(
        frame['solver_retry_runtime_s'], errors='coerce'
    )
    disambiguation_runtime = pd.to_numeric(
        frame['solver_inf_or_unbd_disambiguation_runtime_s'], errors='coerce'
    )
    total_runtime = pd.to_numeric(
        frame['solver_total_runtime_s'], errors='coerce'
    )
    reported_runtime = pd.to_numeric(
        frame['solver_runtime_s'], errors='coerce'
    )
    runtime_accounting_valid = bool(
        bool((initial_runtime >= 0.0).all())
        and bool((retry_runtime >= 0.0).all())
        and bool((disambiguation_runtime >= 0.0).all())
        and np.allclose(
            total_runtime.to_numpy(dtype=float),
            (
                initial_runtime + retry_runtime + disambiguation_runtime
            ).to_numpy(dtype=float),
            rtol=0.0,
            atol=1e-12,
        )
        and np.allclose(
            reported_runtime.to_numpy(dtype=float),
            total_runtime.to_numpy(dtype=float),
            rtol=0.0,
            atol=1e-12,
        )
        and bool((retry_runtime.loc[numeric_retry_used == 0] == 0.0).all())
        and bool((
            disambiguation_runtime.loc[disambiguation_used == 0] == 0.0
        ).all())
    )
    return {
        'provenance_valid': provenance_valid,
        'runtime_accounting_valid': runtime_accounting_valid,
    }


def attach_decision_diagnostic_metrics(
    metrics: pd.DataFrame,
    decision_records: list[dict],
) -> pd.DataFrame:
    enriched = metrics.copy()
    for column in DECISION_SUMMARY_COLUMNS:
        enriched[column] = 0 if column == 'decision_count' else np.nan
    diagnostics = pd.DataFrame(decision_records)
    if diagnostics.empty:
        return enriched
    for case_name, case_diagnostics in diagnostics.groupby('case', sort=False):
        if case_name not in enriched.index:
            raise ValueError(f'Unknown diagnostic case: {case_name}')
        summary = summarize_decision_diagnostics(case_diagnostics)
        for column, value in summary.items():
            enriched.loc[case_name, column] = value
    return enriched


def metric_table_to_records(metrics: pd.DataFrame, segment_date: str) -> pd.DataFrame:
    records = metrics.reset_index().rename(columns={'index': 'case'})
    records.insert(0, 'date', segment_date)
    return records


def sparse_progress(case_name: str, segment_date: str):
    def report(step_idx: int, total_steps: int, timestamp: pd.Timestamp) -> None:
        if step_idx == 0 or step_idx + 1 == total_steps or step_idx % 24 == 0:
            print(
                f'  {segment_date} | {case_name} | '
                f'{step_idx + 1}/{total_steps} @ {timestamp}'
            )
    return report


def _write_csv_exclusive(
    frame: pd.DataFrame,
    path: Path,
    *,
    compression: Optional[str] = None,
) -> None:
    if path.exists():
        raise FileExistsError(f'Refusing existing artifact: {path}')
    frame.to_csv(path, index=False, mode='x', compression=compression)


def _savefig_exclusive(fig, path: Path, *, dpi: int = 180) -> None:
    if path.exists():
        raise FileExistsError(f'Refusing existing figure: {path}')
    with path.open('xb') as handle:
        fig.savefig(handle, format='png', dpi=dpi, bbox_inches='tight')




def _finite_numeric(frame: pd.DataFrame, columns: list[str]) -> bool:
    if any(column not in frame.columns for column in columns):
        return False
    values = frame.loc[:, columns].apply(pd.to_numeric, errors='coerce')
    return bool(np.isfinite(values.to_numpy(dtype=float)).all())


def _validation_row(name: str, expected, actual, passed: bool) -> dict:
    return {
        'check': name,
        'expected': expected,
        'actual': actual,
        'passed': bool(passed),
    }


## Run the sweep


In [ ]:
artifact_paths = {
    'timeseries': OUTPUT_DIR / f'{run_stem}_timeseries.csv.gz',
    'decision_diagnostics': (
        OUTPUT_DIR / f'{run_stem}_decision_diagnostics.csv.gz'
    ),
    'daily_metrics': OUTPUT_DIR / f'{run_stem}_daily_metrics.csv',
    'pooled_metrics': OUTPUT_DIR / f'{run_stem}_pooled_metrics.csv',
    'pareto_table': OUTPUT_DIR / f'{run_stem}_pareto_table.csv',
    'validation': OUTPUT_DIR / f'{run_stem}_validation.csv',
    'errors': OUTPUT_DIR / f'{run_stem}_errors.csv',
    'run_metadata': OUTPUT_DIR / f'{run_stem}_run_metadata.json',
    'weight_response': OUTPUT_DIR / f'{run_stem}_weight_response.png',
    'pareto': OUTPUT_DIR / f'{run_stem}_pareto.png',
    'representative_days': OUTPUT_DIR / f'{run_stem}_representative_days.csv',
}
representative_dir = OUTPUT_DIR / f'{run_stem}_representative_days'
existing_artifacts = [
    str(path) for path in [*artifact_paths.values(), representative_dir]
    if path.exists()
]
if existing_artifacts:
    raise FileExistsError(
        f'Refusing existing intended artifacts: {existing_artifacts}'
    )

if not GUROBI_OK:
    raise RuntimeError('Gurobi is not available; MIQP cases cannot run.')

all_case_results = {case: [] for case in CASE_ORDER}
daily_summary_records = []
timeseries_records = []
error_records = []
decision_diagnostic_records = []

for run_pos, local_idx in enumerate(selected_indices, start=1):
    segment_date = segments[local_idx]['date'].iloc[0].date().isoformat()
    print(f'Running segment {run_pos}/{len(selected_indices)}: {segment_date}')
    diagnostic_record_start = len(decision_diagnostic_records)
    active_case_name = None
    active_controller_objective = None
    active_w_x_slack = np.nan
    try:
        case_results = {}
        active_case_name = 'AC baseline'
        active_controller_objective = 'baseline_ac'
        ac_df = simulate_policy_day(
            'AC baseline', hooks, pv_metric_cfg, day_idx=local_idx,
            policy='ac', **run_window,
        )
        case_results['AC baseline'] = enrich_results(
            ac_df, pv_metric_cfg, 'AC baseline', segment_date,
        )
        active_case_name = 'RBC baseline'
        active_controller_objective = 'baseline_rbc'
        rbc_df = simulate_policy_day(
            'RBC baseline', hooks, pv_metric_cfg, day_idx=local_idx,
            policy='rbc', **run_window,
        )
        case_results['RBC baseline'] = enrich_results(
            rbc_df, pv_metric_cfg, 'RBC baseline', segment_date,
        )

        for (
            case_name,
            case_cfg,
            controller_objective,
            forecast_source,
            weight,
        ) in MIQP_CASE_SPECS:
            active_case_name = case_name
            active_controller_objective = controller_objective
            active_w_x_slack = float(weight)
            hooks.reset_cache()
            miqp_df = simulate_day(
                day_idx=local_idx,
                hooks=hooks,
                cfg=case_cfg,
                init_state_cols=hooks.state_cols,
                progress_fn=sparse_progress(case_name, segment_date),
                forecast_source='observed',
                forecast_case_name=case_name,
                forecast_controller_objective=controller_objective,
                decision_diagnostic_records=decision_diagnostic_records,
                **run_window,
            )
            case_results[case_name] = enrich_results(
                miqp_df,
                pv_metric_cfg,
                case_name,
                segment_date,
                forecast_source=forecast_source,
                controller_objective=controller_objective,
                w_x_slack=weight,
            )

        active_case_name = None
        active_controller_objective = None
        active_w_x_slack = np.nan
        daily_metrics = attach_decision_diagnostic_metrics(
            build_metric_table(case_results),
            decision_diagnostic_records[diagnostic_record_start:],
        )
        daily_summary_records.append(
            metric_table_to_records(daily_metrics, segment_date)
        )
        for case_name, frame in case_results.items():
            all_case_results[case_name].append(frame)
            timeseries_records.append(frame.reset_index())
    except Exception as exc:
        error_records.append({
            'date': segment_date,
            'segment_index': local_idx,
            'case': active_case_name,
            'controller_objective': active_controller_objective,
            'w_x_slack': active_w_x_slack,
            'error_type': type(exc).__name__,
            'error': repr(exc),
        })
        print(f'ERROR on {segment_date}: {exc!r}')

errors_df = pd.DataFrame(
    error_records,
    columns=[
        'date', 'segment_index', 'case', 'controller_objective', 'w_x_slack',
        'error_type', 'error',
    ],
)
if len(errors_df):
    _write_csv_exclusive(errors_df, artifact_paths['errors'])
    failure_validation = pd.DataFrame([
        _validation_row('no_error_rows', 0, len(errors_df), False)
    ])
    _write_csv_exclusive(failure_validation, artifact_paths['validation'])
    raise RuntimeError(
        f"Sweep recorded {len(errors_df)} date/solver errors; "
        f"see {artifact_paths['errors']}"
    )

if not daily_summary_records or not timeseries_records:
    raise RuntimeError('No complete sweep rows were produced.')

missing_cases = [
    case for case in CASE_ORDER if not all_case_results.get(case)
]
if missing_cases:
    raise RuntimeError(f'Missing completed results for cases: {missing_cases}')

daily_metrics_df = pd.concat(daily_summary_records, ignore_index=True)
all_timeseries_df = pd.concat(timeseries_records, ignore_index=True)
decision_diagnostics_df = pd.DataFrame(decision_diagnostic_records)
expected_case_count = len(CASE_ORDER)
expected_miqp_case_count = len(MIQP_CASE_SPECS)

if window_mode == 'full':
    expected_date_count = 22
    expected_daily_rows = 22 * expected_case_count
    expected_timeseries_rows = 22 * expected_case_count * 690
    expected_miqp_decisions = 22 * expected_miqp_case_count * 138
    expected_minutes_per_case_date = 690
    expected_decisions_per_miqp_case_date = 138
else:
    expected_date_count = 1
    expected_daily_rows = expected_case_count
    expected_timeseries_rows = expected_case_count * 90
    expected_miqp_decisions = expected_miqp_case_count * 18
    expected_minutes_per_case_date = 90
    expected_decisions_per_miqp_case_date = 18


## Validate and pool results


In [ ]:
required_timeseries_columns = {
    'ts', 'date', 'case', 'controller_objective', 'forecast_source',
    'w_x_slack', 'z', 'pfc_on', 'T_mean', 'T_out', 'I_solar',
    'hvac_kw', 'pv_kw', 'grid_kw', 'export_kw', 'self_kw', *AC_STATE_COLS,
}
required_daily_columns = {
    'date', 'case', 'controller_objective', 'w_x_slack',
    'zone_degree_minutes_above_30', 'post5_zone_degree_minutes_above_30',
    'any_zone_minutes_above_30', 'post5_any_zone_minutes_above_30',
    'max_zone_temp_c', 'hvac_kwh', 'pv_available_kwh', 'grid_kwh',
    'export_kwh', 'self_consumed_kwh', 'self_consumption_ratio',
    'self_sufficiency_ratio', 'peak_hvac_kw', 'peak_grid_kw',
    'morning_peak_available', 'morning_peak_observed_minutes',
    'morning_peak_expected_minutes', 'morning_peak_hvac_kw',
    'morning_peak_grid_kw', 'evening_peak_available',
    'evening_peak_observed_minutes', 'evening_peak_expected_minutes',
    'evening_peak_hvac_kw', 'evening_peak_grid_kw',
    'energy_shedding_vs_rbc_kwh', 'energy_shedding_vs_rbc_pct',
    'pfc_on_fraction', 'window_open_fraction', 'switch_count',
    'rain_safety_violation_minutes', 'decision_count',
    'predicted_upper_slack_sum',
    'predicted_upper_slack_positive_fraction',
    'numeric_retry_count', 'numeric_retry_fraction',
    'inf_or_unbd_disambiguation_count',
    'inf_or_unbd_disambiguation_fraction',
    'solver_initial_runtime_sum_s', 'solver_retry_runtime_sum_s',
    'solver_inf_or_unbd_disambiguation_runtime_sum_s',
    'solver_total_runtime_sum_s',
    'solver_runtime_sum_s', 'solver_runtime_mean_s',
    'solver_runtime_max_s', 'solver_mip_gap_available_fraction',
    'solver_mip_gap_mean', 'solver_mip_gap_max',
}
required_decision_columns = {
    'date', 'case', 'decision_time', 'controller_objective',
    'forecast_source', 'w_x_slack', 'chosen_mode_z', 'previous_mode_z',
    'pfcu_on', 'dwell_lock_steps_before', 'effective_dwell_lock_steps',
    'rain_observed_active', 'rain_force_ac', 'rain_lock_steps_before',
    'rain_lock_steps_after', 'objective_total', 'objective_comfort',
    'objective_du', 'objective_switch', 'objective_energy',
    'objective_x_slack', 'objective_pv', 'predicted_max_temp_c',
    'predicted_upper_slack_sum', 'predicted_upper_slack_positive',
    'solver_initial_status', 'solver_final_status', 'solver_status',
    'numeric_retry_used', 'solver_retry_count',
    'inf_or_unbd_disambiguation_used',
    'solver_inf_or_unbd_disambiguation_count',
    'solver_initial_runtime_s', 'solver_retry_runtime_s',
    'solver_inf_or_unbd_disambiguation_runtime_s',
    'solver_total_runtime_s', 'solver_retry_numeric_focus',
    'solver_runtime_s', 'solver_mip_gap', 'solver_mip_gap_available',
}
missing_required = sorted(
    (required_timeseries_columns - set(all_timeseries_df.columns))
    | (required_daily_columns - set(daily_metrics_df.columns))
    | (required_decision_columns - set(decision_diagnostics_df.columns))
)

timeseries_numeric = [
    *AC_STATE_COLS, 'z', 'pfc_on', 'T_mean', 'T_out', 'I_solar',
    'hvac_kw', 'pv_kw', 'grid_kw', 'export_kw', 'self_kw',
]
daily_numeric = [
    'zone_degree_minutes_above_30', 'post5_zone_degree_minutes_above_30',
    'any_zone_minutes_above_30', 'post5_any_zone_minutes_above_30',
    'max_zone_temp_c', 'hvac_kwh', 'pv_available_kwh', 'grid_kwh',
    'export_kwh', 'self_consumed_kwh', 'peak_hvac_kw', 'peak_grid_kw',
    'morning_peak_available', 'morning_peak_observed_minutes',
    'morning_peak_expected_minutes', 'evening_peak_available',
    'evening_peak_observed_minutes', 'evening_peak_expected_minutes',
    'energy_shedding_vs_rbc_kwh',
    'pfc_on_fraction', 'window_open_fraction', 'switch_count',
    'rain_safety_violation_minutes',
]
decision_numeric = [
    'w_x_slack', 'chosen_mode_z', 'previous_mode_z', 'pfcu_on',
    'dwell_lock_steps_before', 'effective_dwell_lock_steps',
    'rain_observed_active', 'rain_force_ac', 'rain_lock_steps_before',
    'rain_lock_steps_after', 'objective_total', 'objective_comfort',
    'objective_du', 'objective_switch', 'objective_energy',
    'objective_x_slack', 'objective_pv', 'predicted_max_temp_c',
    'predicted_upper_slack_sum', 'predicted_upper_slack_positive',
    'solver_initial_status', 'solver_final_status', 'solver_status',
    'numeric_retry_used', 'solver_retry_count',
    'inf_or_unbd_disambiguation_used',
    'solver_inf_or_unbd_disambiguation_count',
    'solver_initial_runtime_s', 'solver_retry_runtime_s',
    'solver_inf_or_unbd_disambiguation_runtime_s',
    'solver_total_runtime_s', 'solver_runtime_s',
    'solver_mip_gap_available',
]
nonfinite_failures = pd.concat(
    [
        find_nonfinite_numeric_failures(
            all_timeseries_df,
            timeseries_numeric,
            table_name='timeseries',
        ),
        find_nonfinite_numeric_failures(
            daily_metrics_df,
            daily_numeric,
            table_name='daily_metrics',
        ),
        find_nonfinite_numeric_failures(
            decision_diagnostics_df,
            decision_numeric,
            table_name='decision_diagnostics',
        ),
    ],
    ignore_index=True,
)
numeric_finite = not missing_required and nonfinite_failures.empty
numeric_finite_actual = (
    True
    if numeric_finite
    else format_validation_failures(nonfinite_failures)
)

daily_ratio_frame = daily_metrics_df.copy()
daily_ratio_base_columns = {'date', 'controller_objective', 'hvac_kwh'}
if daily_ratio_base_columns.issubset(daily_ratio_frame.columns):
    daily_rbc_hvac_by_date = (
        daily_ratio_frame.loc[
            daily_ratio_frame['controller_objective'] == 'baseline_rbc'
        ]
        .groupby('date', sort=False)['hvac_kwh']
        .first()
    )
    daily_ratio_frame['_daily_rbc_hvac_kwh'] = daily_ratio_frame['date'].map(
        daily_rbc_hvac_by_date
    )
    daily_ratio_frame['_daily_energy_shedding_vs_rbc_kwh'] = (
        daily_ratio_frame['_daily_rbc_hvac_kwh']
        - daily_ratio_frame['hvac_kwh']
    )
daily_ratio_contracts = [
    DenominatorAwareRatioContract(
        'self_consumption_ratio',
        'self_consumed_kwh',
        'pv_available_kwh',
        denominator_guard_gt=0.0,
    ),
    DenominatorAwareRatioContract(
        'self_sufficiency_ratio',
        'self_consumed_kwh',
        'hvac_kwh',
        denominator_guard_gt=0.0,
    ),
    DenominatorAwareRatioContract(
        'energy_shedding_vs_rbc_pct',
        '_daily_energy_shedding_vs_rbc_kwh',
        '_daily_rbc_hvac_kwh',
        denominator_guard_gt=0.05,
        scale=100.0,
    ),
]
daily_ratio_failures = validate_denominator_aware_ratios(
    daily_ratio_frame,
    daily_ratio_contracts,
    table_name='daily_metrics',
)
daily_ratio_contracts_valid = daily_ratio_failures.empty
daily_ratio_contracts_actual = (
    True
    if daily_ratio_contracts_valid
    else format_validation_failures(daily_ratio_failures)
)
pre_analysis_validation = pd.DataFrame([
    _validation_row(
        'required_columns_present', [], missing_required,
        not missing_required,
    ),
    _validation_row(
        'required_numeric_values_finite', True, numeric_finite_actual,
        numeric_finite,
    ),
    _validation_row(
        'denominator_aware_daily_ratio_contracts', True,
        daily_ratio_contracts_actual,
        daily_ratio_contracts_valid,
    ),
])
if not bool(pre_analysis_validation['passed'].all()):
    _write_csv_exclusive(errors_df, artifact_paths['errors'])
    _write_csv_exclusive(
        pre_analysis_validation, artifact_paths['validation']
    )
    failed_checks = pre_analysis_validation.loc[
        ~pre_analysis_validation['passed'], 'check'
    ].tolist()
    raise RuntimeError(
        f'Missing/non-finite sweep data; validation failed: {failed_checks}'
    )

gap_available = pd.to_numeric(
    decision_diagnostics_df['solver_mip_gap_available'], errors='coerce'
)
gap_values = pd.to_numeric(
    decision_diagnostics_df['solver_mip_gap'], errors='coerce'
)
gap_availability_binary = bool(gap_available.isin([0, 1]).all())
available_gap_values = gap_values.loc[gap_available == 1]
available_gap_valid = bool(
    np.isfinite(available_gap_values.to_numpy(dtype=float)).all()
    and bool((gap_values.loc[gap_available == 1] >= 0.0).all())
)
unavailable_gap_nan = bool(
    gap_values.loc[gap_available == 0].isna().all()
)

solver_recovery_validation = validate_solver_recovery_provenance(
    decision_diagnostics_df, GRB
)
solver_recovery_provenance_valid = solver_recovery_validation[
    'provenance_valid'
]
solver_recovery_runtime_accounting_valid = solver_recovery_validation[
    'runtime_accounting_valid'
]

daily_miqp_metrics = daily_metrics_df.loc[
    daily_metrics_df['controller_objective'].isin(['no_pv', 'onsite_pv'])
]
daily_diagnostic_numeric = [
    'decision_count', 'predicted_upper_slack_sum',
    'predicted_upper_slack_positive_fraction',
    'numeric_retry_count', 'numeric_retry_fraction',
    'inf_or_unbd_disambiguation_count',
    'inf_or_unbd_disambiguation_fraction',
    'solver_initial_runtime_sum_s', 'solver_retry_runtime_sum_s',
    'solver_inf_or_unbd_disambiguation_runtime_sum_s',
    'solver_total_runtime_sum_s', 'solver_runtime_sum_s',
    'solver_runtime_mean_s', 'solver_runtime_max_s',
    'solver_mip_gap_available_fraction',
]
daily_diagnostic_summaries_finite = _finite_numeric(
    daily_miqp_metrics, daily_diagnostic_numeric
)

miqp_timeseries = all_timeseries_df.loc[
    all_timeseries_df['controller_objective'].isin(['no_pv', 'onsite_pv'])
].copy()
rbc_timeseries = all_timeseries_df.loc[
    all_timeseries_df['controller_objective'] == 'baseline_rbc'
]
rbc_hvac_kwh = float(rbc_timeseries['hvac_kw'].sum()) / 60.0
if rbc_hvac_kwh <= 0.05:
    raise RuntimeError(
        f'Pooled RBC HVAC energy is too small for shedding: {rbc_hvac_kwh}'
    )
pooled_records = []
for controller_objective in ('no_pv', 'onsite_pv'):
    for weight in SWEEP_WEIGHTS:
        pooled_case = miqp_timeseries.loc[
            (miqp_timeseries['controller_objective'] == controller_objective)
            & (miqp_timeseries['w_x_slack'] == weight)
        ].copy()
        if pooled_case.empty:
            raise RuntimeError(
                f'Missing pooled rows for {controller_objective}, wx={weight}'
            )
        thermal = temperature_metrics(
            pooled_case, AC_STATE_COLS, bound_c=30.0
        )
        matching_daily = daily_metrics_df.loc[
            (daily_metrics_df['controller_objective'] == controller_objective)
            & (daily_metrics_df['w_x_slack'] == weight)
        ]
        hvac_kwh = float(pooled_case['hvac_kw'].sum()) / 60.0
        pv_available_kwh = float(pooled_case['pv_kw'].sum()) / 60.0
        grid_kwh = float(pooled_case['grid_kw'].sum()) / 60.0
        export_kwh = float(pooled_case['export_kw'].sum()) / 60.0
        self_consumed_kwh = float(pooled_case['self_kw'].sum()) / 60.0
        energy_shedding_vs_rbc_kwh = rbc_hvac_kwh - hvac_kwh
        pooled_peak_windows = peak_window_metrics(pooled_case)
        pooled_diagnostics = decision_diagnostics_df.loc[
            (decision_diagnostics_df['controller_objective'] == controller_objective)
            & (decision_diagnostics_df['w_x_slack'] == weight)
        ]
        pooled_diagnostic_summary = summarize_decision_diagnostics(
            pooled_diagnostics
        )
        pooled_records.append({
            'controller_objective': controller_objective,
            'w_x_slack': float(weight),
            **thermal,
            'post5_any_zone_minutes_above_30': int(
                matching_daily['post5_any_zone_minutes_above_30'].sum()
            ),
            'post5_zone_degree_minutes_above_30': float(
                matching_daily['post5_zone_degree_minutes_above_30'].sum()
            ),
            'hvac_kwh': hvac_kwh,
            'pv_available_kwh': pv_available_kwh,
            'grid_kwh': grid_kwh,
            'export_kwh': export_kwh,
            'self_consumed_kwh': self_consumed_kwh,
            'self_consumption_ratio': (
                self_consumed_kwh / pv_available_kwh
                if pv_available_kwh > 0 else np.nan
            ),
            'self_sufficiency_ratio': (
                self_consumed_kwh / hvac_kwh if hvac_kwh > 0 else np.nan
            ),
            'energy_shedding_vs_rbc_kwh': energy_shedding_vs_rbc_kwh,
            'energy_shedding_vs_rbc_pct': (
                100.0 * energy_shedding_vs_rbc_kwh / rbc_hvac_kwh
            ),
            'peak_hvac_kw': float(pooled_case['hvac_kw'].max()),
            'peak_grid_kw': float(pooled_case['grid_kw'].max()),
            **pooled_peak_windows,
            'pfc_on_fraction': float(
                pooled_case['pfc_on'].astype(int).mean()
            ),
            'window_open_fraction': float(
                (1 - pooled_case['z'].astype(int)).mean()
            ),
            'switch_count': int(matching_daily['switch_count'].sum()),
            **pooled_diagnostic_summary,
        })

pooled_metrics_df = pd.DataFrame(pooled_records).sort_values(
    ['controller_objective', 'w_x_slack'], kind='stable'
).reset_index(drop=True)
pareto_table_df, selection_summary = build_balanced_selection(
    pooled_metrics_df
)

cases_per_date = daily_metrics_df.groupby('date')['case'].nunique()
minutes_per_case_date = all_timeseries_df.groupby(['date', 'case']).size()
decisions_per_case_date = decision_diagnostics_df.groupby(
    ['date', 'case']
).size()
accepted_solver_statuses = {GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT}
solver_status_values = set(
    decision_diagnostics_df['solver_status'].astype(int).unique().tolist()
)
rain_lockout_violations = int(
    daily_metrics_df['rain_safety_violation_minutes'].sum()
)
dwell_lock_violations = int((
    (decision_diagnostics_df['effective_dwell_lock_steps'] > 0)
    & (decision_diagnostics_df['rain_force_ac'] == 0)
    & (
        decision_diagnostics_df['chosen_mode_z']
        != decision_diagnostics_df['previous_mode_z']
    )
).sum())
pfcu_ac_violations = int((
    (decision_diagnostics_df['chosen_mode_z'] == 1)
    & (decision_diagnostics_df['pfcu_on'] != 0)
).sum())

peak_flag_columns = ['morning_peak_available', 'evening_peak_available']
peak_flags = pd.concat(
    [daily_metrics_df[peak_flag_columns], pooled_metrics_df[peak_flag_columns]],
    ignore_index=True,
).apply(pd.to_numeric, errors='coerce')
peak_window_flags_binary = bool(
    peak_flags.isin([0, 1]).to_numpy().all()
)
full_morning_peak_metrics_available = bool(
    window_mode != 'full'
    or (
        (daily_metrics_df['morning_peak_available'] == 1).all()
        and (pooled_metrics_df['morning_peak_available'] == 1).all()
        and _finite_numeric(
            daily_metrics_df,
            ['morning_peak_hvac_kw', 'morning_peak_grid_kw'],
        )
        and _finite_numeric(
            pooled_metrics_df,
            ['morning_peak_hvac_kw', 'morning_peak_grid_kw'],
        )
    )
)
full_evening_peak_metrics_available = bool(
    window_mode != 'full'
    or (
        (daily_metrics_df['evening_peak_available'] == 1).all()
        and (pooled_metrics_df['evening_peak_available'] == 1).all()
        and _finite_numeric(
            daily_metrics_df,
            ['evening_peak_hvac_kw', 'evening_peak_grid_kw'],
        )
        and _finite_numeric(
            pooled_metrics_df,
            ['evening_peak_hvac_kw', 'evening_peak_grid_kw'],
        )
    )
)
smoke_morning_peak_metrics_unavailable = bool(
    window_mode != 'smoke'
    or (
        (daily_metrics_df['morning_peak_available'] == 0).all()
        and (pooled_metrics_df['morning_peak_available'] == 0).all()
        and daily_metrics_df[
            ['morning_peak_hvac_kw', 'morning_peak_grid_kw']
        ].isna().to_numpy().all()
        and pooled_metrics_df[
            ['morning_peak_hvac_kw', 'morning_peak_grid_kw']
        ].isna().to_numpy().all()
    )
)
smoke_evening_peak_metrics_unavailable = bool(
    window_mode != 'smoke'
    or (
        (daily_metrics_df['evening_peak_available'] == 0).all()
        and (pooled_metrics_df['evening_peak_available'] == 0).all()
        and daily_metrics_df[
            ['evening_peak_hvac_kw', 'evening_peak_grid_kw']
        ].isna().to_numpy().all()
        and pooled_metrics_df[
            ['evening_peak_hvac_kw', 'evening_peak_grid_kw']
        ].isna().to_numpy().all()
    )
)
pooled_reporting_numeric = [
    'energy_shedding_vs_rbc_kwh', 'energy_shedding_vs_rbc_pct',
    'morning_peak_observed_minutes', 'morning_peak_expected_minutes',
    'evening_peak_observed_minutes', 'evening_peak_expected_minutes',
    'pfc_on_fraction', 'decision_count', 'predicted_upper_slack_sum',
    'predicted_upper_slack_positive_fraction',
    'numeric_retry_count', 'numeric_retry_fraction',
    'inf_or_unbd_disambiguation_count',
    'inf_or_unbd_disambiguation_fraction',
    'solver_initial_runtime_sum_s', 'solver_retry_runtime_sum_s',
    'solver_inf_or_unbd_disambiguation_runtime_sum_s',
    'solver_total_runtime_sum_s', 'solver_runtime_sum_s',
    'solver_runtime_mean_s', 'solver_runtime_max_s',
    'solver_mip_gap_available_fraction',
]
pooled_reporting_metrics_finite = _finite_numeric(
    pooled_metrics_df, pooled_reporting_numeric
)

validation_records = [
    _validation_row('no_error_rows', 0, len(errors_df), len(errors_df) == 0),
    _validation_row(
        'cpu_device_selected', 'cpu', str(hooks.device),
        str(hooks.device) == 'cpu',
    ),
    _validation_row(
        'three_sweep_weights', [100.0, 1000.0, 10000.0],
        list(SWEEP_WEIGHTS),
        tuple(SWEEP_WEIGHTS) == (100.0, 1000.0, 10000.0),
    ),
    _validation_row(
        'selected_date_count', expected_date_count,
        len(selected_dates), len(selected_dates) == expected_date_count,
    ),
    _validation_row(
        'daily_row_count', expected_daily_rows,
        len(daily_metrics_df), len(daily_metrics_df) == expected_daily_rows,
    ),
    _validation_row(
        'timeseries_row_count', expected_timeseries_rows,
        len(all_timeseries_df), len(all_timeseries_df) == expected_timeseries_rows,
    ),
    _validation_row(
        'miqp_decision_count', expected_miqp_decisions,
        len(decision_diagnostics_df),
        len(decision_diagnostics_df) == expected_miqp_decisions,
    ),
    _validation_row(
        'daily_cases_per_date', expected_case_count,
        sorted(cases_per_date.unique().tolist()),
        len(cases_per_date) == expected_date_count
        and bool((cases_per_date == expected_case_count).all()),
    ),
    _validation_row(
        'timeseries_rows_per_case_date', expected_minutes_per_case_date,
        sorted(minutes_per_case_date.unique().tolist()),
        len(minutes_per_case_date) == expected_daily_rows
        and bool((minutes_per_case_date == expected_minutes_per_case_date).all()),
    ),
    _validation_row(
        'decision_rows_per_miqp_case_date',
        expected_decisions_per_miqp_case_date,
        sorted(decisions_per_case_date.unique().tolist()),
        len(decisions_per_case_date) == expected_date_count * expected_miqp_case_count
        and bool((decisions_per_case_date == expected_decisions_per_miqp_case_date).all()),
    ),
    _validation_row(
        'required_columns_present', [], missing_required,
        not missing_required,
    ),
    _validation_row(
        'required_numeric_values_finite', True,
        numeric_finite_actual, numeric_finite,
    ),
    _validation_row(
        'denominator_aware_daily_ratio_contracts', True,
        daily_ratio_contracts_actual,
        daily_ratio_contracts_valid,
    ),
    _validation_row(
        'solver_statuses_accepted', sorted(accepted_solver_statuses),
        sorted(solver_status_values),
        solver_status_values.issubset(accepted_solver_statuses),
    ),
    _validation_row(
        'solver_mip_gap_availability_binary', True,
        gap_availability_binary, gap_availability_binary,
    ),
    _validation_row(
        'available_solver_mip_gap_finite_nonnegative', True,
        available_gap_valid, available_gap_valid,
    ),
    _validation_row(
        'unavailable_solver_mip_gap_is_nan', True,
        unavailable_gap_nan, unavailable_gap_nan,
    ),
    _validation_row(
        'solver_recovery_provenance_valid', True,
        solver_recovery_provenance_valid, solver_recovery_provenance_valid,
    ),
    _validation_row(
        'solver_recovery_runtime_accounting_valid', True,
        solver_recovery_runtime_accounting_valid,
        solver_recovery_runtime_accounting_valid,
    ),
    _validation_row(
        'daily_diagnostic_summaries_finite', True,
        daily_diagnostic_summaries_finite,
        daily_diagnostic_summaries_finite,
    ),
    _validation_row(
        'pooled_reporting_metrics_finite', True,
        pooled_reporting_metrics_finite,
        pooled_reporting_metrics_finite,
    ),
    _validation_row(
        'peak_window_availability_flags_binary', True,
        peak_window_flags_binary, peak_window_flags_binary,
    ),
    _validation_row(
        'morning_peak_metrics_available_if_full', True,
        full_morning_peak_metrics_available,
        full_morning_peak_metrics_available,
    ),
    _validation_row(
        'evening_peak_metrics_available_if_full', True,
        full_evening_peak_metrics_available,
        full_evening_peak_metrics_available,
    ),
    _validation_row(
        'smoke_morning_peak_metrics_unavailable', True,
        smoke_morning_peak_metrics_unavailable,
        smoke_morning_peak_metrics_unavailable,
    ),
    _validation_row(
        'smoke_evening_peak_metrics_unavailable', True,
        smoke_evening_peak_metrics_unavailable,
        smoke_evening_peak_metrics_unavailable,
    ),
    _validation_row(
        'rain_lockout_safety', 0, rain_lockout_violations,
        rain_lockout_violations == 0,
    ),
    _validation_row(
        'dwell_lock_safety', 0, dwell_lock_violations,
        dwell_lock_violations == 0,
    ),
    _validation_row(
        'pfcu_disabled_in_ac_mode', 0, pfcu_ac_violations,
        pfcu_ac_violations == 0,
    ),
]
validation_df = pd.DataFrame(validation_records)

if not bool(validation_df['passed'].all()):
    _write_csv_exclusive(errors_df, artifact_paths['errors'])
    _write_csv_exclusive(validation_df, artifact_paths['validation'])
    failed_checks = validation_df.loc[
        ~validation_df['passed'], 'check'
    ].tolist()
    raise RuntimeError(f'Pre-report validation failed: {failed_checks}')

_write_csv_exclusive(
    all_timeseries_df,
    artifact_paths['timeseries'],
    compression='gzip',
)
_write_csv_exclusive(
    decision_diagnostics_df,
    artifact_paths['decision_diagnostics'],
    compression='gzip',
)
_write_csv_exclusive(daily_metrics_df, artifact_paths['daily_metrics'])
_write_csv_exclusive(pooled_metrics_df, artifact_paths['pooled_metrics'])
_write_csv_exclusive(pareto_table_df, artifact_paths['pareto_table'])
_write_csv_exclusive(errors_df, artifact_paths['errors'])

display_columns = [
    'controller_objective', 'w_x_slack',
    'zone_degree_minutes_above_30', 'any_zone_minutes_above_30',
    'max_zone_temp_c', 'hvac_kwh', 'self_consumption_ratio',
    'self_sufficiency_ratio', 'energy_shedding_vs_rbc_kwh',
    'energy_shedding_vs_rbc_pct', 'morning_peak_available',
    'morning_peak_observed_minutes', 'morning_peak_expected_minutes',
    'morning_peak_hvac_kw', 'morning_peak_grid_kw',
    'evening_peak_available', 'evening_peak_observed_minutes',
    'evening_peak_expected_minutes',
    'evening_peak_hvac_kw', 'evening_peak_grid_kw', 'pfc_on_fraction',
    'predicted_upper_slack_sum',
    'predicted_upper_slack_positive_fraction', 'solver_runtime_sum_s',
    'solver_mip_gap_available_fraction', 'solver_mip_gap_mean',
    'window_open_fraction', 'switch_count',
]
pooled_metrics_df[display_columns].round(4)


In [ ]:
# Weight response across the two controller objectives.
controller_colors = {'no_pv': 'tab:blue', 'onsite_pv': 'tab:orange'}
fig, axes = plt.subplots(4, 2, figsize=(16, 18))
response_specs = [
    ('any_zone_minutes_above_30', 'Any-zone violation minutes'),
    ('zone_degree_minutes_above_30', 'Zone degree-minutes above 30 C'),
    ('max_zone_temp_c', 'Maximum zone temperature (C)'),
    ('hvac_kwh', 'HVAC energy (kWh)'),
    ('window_open_fraction', 'Window-open fraction'),
    ('switch_count', 'Mode switches'),
]
for axis, (metric, title) in zip(axes.ravel(), response_specs):
    for objective in ('no_pv', 'onsite_pv'):
        data = pooled_metrics_df.loc[
            pooled_metrics_df['controller_objective'] == objective
        ].sort_values('w_x_slack')
        axis.plot(
            data['w_x_slack'], data[metric], marker='o',
            color=controller_colors[objective], label=objective,
        )
        for _, row in data.iterrows():
            axis.annotate(
                f"{int(row['w_x_slack'])}",
                (row['w_x_slack'], row[metric]),
                xytext=(4, 4), textcoords='offset points', fontsize=8,
            )
    axis.set_xscale('log')
    axis.set_title(title)
    axis.set_xlabel('w_x_slack')
    axis.grid(True, alpha=0.25)
    axis.legend()

ratio_axis = axes.ravel()[6]
pv_response = pooled_metrics_df.loc[
    pooled_metrics_df['controller_objective'] == 'onsite_pv'
].sort_values('w_x_slack')
for metric, label, marker in (
    ('self_consumption_ratio', 'SC', 'o'),
    ('self_sufficiency_ratio', 'SS', 's'),
):
    ratio_axis.plot(
        pv_response['w_x_slack'], pv_response[metric],
        marker=marker, label=label,
    )
    for _, row in pv_response.iterrows():
        ratio_axis.annotate(
            f"{int(row['w_x_slack'])}",
            (row['w_x_slack'], row[metric]),
            xytext=(4, -10), textcoords='offset points', fontsize=8,
        )
ratio_axis.set_xscale('log')
ratio_axis.set_ylim(0, 1.08)
ratio_axis.set_title(
    'MPC-PV self-consumption / self-sufficiency', pad=12
)
ratio_axis.set_xlabel('w_x_slack')
ratio_axis.grid(True, alpha=0.25)
ratio_axis.legend()
axes.ravel()[7].axis('off')
fig.suptitle('Weight response', y=1.01)
fig.tight_layout()
_savefig_exclusive(fig, artifact_paths['weight_response'])
plt.close(fig)

# Three-panel Pareto view: ordinary MPC, MPC-PV, and combined normalized.
def _annotate_pareto_weights(axis, data, x_col, y_col):
    grouped = data.groupby([x_col, y_col], sort=False, dropna=False)
    for (x_value, y_value), points in grouped:
        weights = sorted(
            int(weight) for weight in points['w_x_slack'].tolist()
        )
        label = ', '.join(str(weight) for weight in weights)
        if len(weights) > 1:
            label += ' (tied)'
        axis.annotate(
            label,
            (x_value, y_value),
            xytext=(4, 4), textcoords='offset points', fontsize=8,
        )


fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for axis, objective, title in zip(
    axes[:2],
    ('no_pv', 'onsite_pv'),
    ('Ordinary MPC', 'MPC-PV'),
):
    data = pareto_table_df.loc[
        pareto_table_df['controller_objective'] == objective
    ].sort_values('w_x_slack')
    frontier = data.loc[data['controller_pareto']]
    axis.scatter(
        data['thermal_pareto_display'], data['energy_pareto_display'],
        color=controller_colors[objective], alpha=0.45,
    )
    axis.plot(
        frontier['thermal_pareto_display'],
        frontier['energy_pareto_display'],
        color=controller_colors[objective], marker='o',
    )
    _annotate_pareto_weights(
        axis, data, 'thermal_pareto_display', 'energy_pareto_display'
    )
    knee = selection_summary['individual'][objective]['knee_weight']
    if knee is not None:
        marked = data.loc[data['w_x_slack'] == knee].iloc[0]
        axis.scatter(
            [marked['thermal_pareto_display']],
            [marked['energy_pareto_display']],
            marker='*', s=180, color='black', label=f'knee={int(knee)}',
        )
        axis.legend()
    if selection_summary['individual'][objective]['no_material_response']:
        axis.text(
            0.03, 0.95, 'No material response within tolerances',
            transform=axis.transAxes, va='top', fontsize=9,
        )
    axis.set_title(title)
    axis.set_xlabel('Zone degree-minutes above 30 C')
    axis.set_ylabel('HVAC energy (kWh)')
    axis.grid(True, alpha=0.25)

combined = (
    pareto_table_df[
        ['w_x_slack', 'combined_thermal_normalized',
         'combined_energy_normalized', 'combined_pareto']
    ]
    .drop_duplicates('w_x_slack')
    .sort_values('w_x_slack')
)
combined_frontier = combined.loc[combined['combined_pareto']]
axes[2].scatter(
    combined['combined_thermal_normalized'],
    combined['combined_energy_normalized'],
    color='tab:green', alpha=0.45,
)
axes[2].plot(
    combined_frontier['combined_thermal_normalized'],
    combined_frontier['combined_energy_normalized'],
    color='tab:green', marker='o',
)
_annotate_pareto_weights(
    axes[2], combined,
    'combined_thermal_normalized', 'combined_energy_normalized',
)
common_knee = selection_summary['common_knee_weight']
if common_knee is not None:
    marked = combined.loc[combined['w_x_slack'] == common_knee].iloc[0]
    axes[2].scatter(
        [marked['combined_thermal_normalized']],
        [marked['combined_energy_normalized']],
        marker='*', s=180, color='black',
        label=f'common knee={int(common_knee)}',
    )
    axes[2].legend()
axes[2].set_title('Combined normalized')
axes[2].set_xlabel('Normalized thermal burden')
axes[2].set_ylabel('Normalized HVAC energy')
axes[2].grid(True, alpha=0.25)
fig.suptitle('Three-panel Pareto', y=1.02)
fig.tight_layout()
_savefig_exclusive(fig, artifact_paths['pareto'])
plt.close(fig)


## Representative days and portable run metadata


In [ ]:
weight_100_daily = daily_metrics_df.loc[
    daily_metrics_df['controller_objective'].isin(['no_pv', 'onsite_pv'])
    & (daily_metrics_df['w_x_slack'] == SWEEP_WEIGHTS[0])
]
hottest_dates = (
    weight_100_daily.groupby('date')['zone_degree_minutes_above_30']
    .max()
    .nlargest(min(3, expected_date_count))
    .index.tolist()
)
representative_records = [
    {
        'date': date,
        'controller_objective': 'both',
        'reason': 'hottest wx=100 day',
        'mode_changes_vs_wx_100': np.nan,
        'selected_weight': selection_summary['common_knee_weight'],
    }
    for date in hottest_dates
]

common_knee = selection_summary['common_knee_weight']
if common_knee is not None:
    for date in selected_dates:
        for objective in ('no_pv', 'onsite_pv'):
            base = miqp_timeseries.loc[
                (miqp_timeseries['date'] == date)
                & (miqp_timeseries['controller_objective'] == objective)
                & (miqp_timeseries['w_x_slack'] == SWEEP_WEIGHTS[0]),
                ['ts', 'z'],
            ].sort_values('ts')
            selected = miqp_timeseries.loc[
                (miqp_timeseries['date'] == date)
                & (miqp_timeseries['controller_objective'] == objective)
                & (miqp_timeseries['w_x_slack'] == common_knee),
                ['ts', 'z'],
            ].sort_values('ts')
            aligned = base.merge(
                selected, on='ts', suffixes=('_wx100', '_selected'),
                validate='one_to_one',
            )
            if len(aligned) != expected_minutes_per_case_date:
                raise RuntimeError(
                    f'Representative-day mode alignment failed for {date}, '
                    f'{objective}: {len(aligned)} rows'
                )
            mode_changes = int(
                (aligned['z_wx100'] != aligned['z_selected']).sum()
            )
            if mode_changes >= 35:
                representative_records.append({
                    'date': date,
                    'controller_objective': objective,
                    'reason': 'mode changes vs wx=100 >= 35 minutes',
                    'mode_changes_vs_wx_100': mode_changes,
                    'selected_weight': common_knee,
                })
else:
    for record in representative_records:
        record['reason'] += '; no common knee / selected-weight comparison'

representative_days_df = pd.DataFrame(representative_records).sort_values(
    ['date', 'controller_objective', 'reason'], kind='stable'
).reset_index(drop=True)
representative_dates = representative_days_df['date'].drop_duplicates().tolist()
representative_dir.mkdir(parents=False, exist_ok=False)
_write_csv_exclusive(
    representative_days_df,
    artifact_paths['representative_days'],
)

for date in representative_dates:
    fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True)
    for pair_index, objective in enumerate(('no_pv', 'onsite_pv')):
        baseline = miqp_timeseries.loc[
            (miqp_timeseries['date'] == date)
            & (miqp_timeseries['controller_objective'] == objective)
            & (miqp_timeseries['w_x_slack'] == SWEEP_WEIGHTS[0])
        ].sort_values('ts')
        if baseline.empty:
            raise RuntimeError(
                f'Missing wx=100 representative rows for {date}, {objective}'
            )
        baseline_ts = pd.to_datetime(baseline['ts'])
        baseline_max = baseline[AC_STATE_COLS].max(axis=1)
        temp_axis = axes[pair_index * 2]
        mode_axis = axes[pair_index * 2 + 1]
        temp_axis.plot(
            baseline_ts, baseline_max,
            label=f'{objective} wx=100 max-zone T',
            color=controller_colors[objective], linestyle='--',
        )
        mode_axis.step(
            baseline_ts, baseline['z'], where='post',
            label=f'{objective} wx=100 mode',
            color=controller_colors[objective], linestyle='--',
        )
        mode_axis.step(
            baseline_ts, baseline['pfc_on'], where='post',
            label=f'{objective} wx=100 PFCU',
            color='tab:gray', linestyle=':',
        )
        if common_knee is not None:
            selected = miqp_timeseries.loc[
                (miqp_timeseries['date'] == date)
                & (miqp_timeseries['controller_objective'] == objective)
                & (miqp_timeseries['w_x_slack'] == common_knee)
            ].sort_values('ts')
            if selected.empty:
                raise RuntimeError(
                    f'Missing selected representative rows for '
                    f'{date}, {objective}, wx={common_knee}'
                )
            selected_ts = pd.to_datetime(selected['ts'])
            selected_max = selected[AC_STATE_COLS].max(axis=1)
            temp_axis.plot(
                selected_ts, selected_max,
                label=f'{objective} wx={int(common_knee)} max-zone T',
                color=controller_colors[objective],
            )
            mode_axis.step(
                selected_ts, selected['z'], where='post',
                label=f'{objective} wx={int(common_knee)} mode',
                color=controller_colors[objective],
            )
            mode_axis.step(
                selected_ts, selected['pfc_on'], where='post',
                label=f'{objective} wx={int(common_knee)} PFCU',
                color='black', linestyle=':',
            )
        temp_axis.axhline(30.0, color='red', linestyle=':', label='30 C bound')
        temp_axis.set_ylabel('Max zone T (C)')
        temp_axis.grid(True, alpha=0.25)
        temp_axis.legend(loc='best', ncol=2)
        mode_axis.set_ylabel('Mode / PFCU')
        mode_axis.set_yticks([0, 1])
        mode_axis.grid(True, alpha=0.25)
        mode_axis.legend(loc='best', ncol=2)
    axes[-1].set_xlabel('Time')
    fig.suptitle(f'Representative day {date}', y=1.01)
    fig.tight_layout()
    _savefig_exclusive(fig, representative_dir / f'{date}.png')
    plt.close(fig)


validation_passed = bool(validation_df["passed"].all())
run_metadata = {
    "study": "Temperature upper-slack penalty sensitivity",
    "input_paths": {
        "observed_data": str(L14_DATA_PATH.relative_to(REPO_ROOT)),
        "ac_model": str(AC_MODEL_PATH.relative_to(REPO_ROOT)),
        "nv_model": str(NV_MODEL_PATH.relative_to(REPO_ROOT)),
        "pv_coefficients": str(PV_COEFFICIENT_PATH.relative_to(REPO_ROOT)),
    },
    "copied_input_manifest": {
        "l14_merged_data_with_rain.csv": {
            "copy": str(L14_DATA_PATH.relative_to(REPO_ROOT))
        }
    },
    "runtime": {
        "device": str(hooks.device),
        "cuda_available": bool(torch.cuda.is_available()),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
    "varied_configuration": {"parameter": "w_x_slack", "weights": list(SWEEP_WEIGHTS)},
    "fixed_configuration": FIXED_MPC_KWARGS,
    "selected_dates": selected_dates,
    "window_mode": window_mode,
    "validation_passed": validation_passed,
}
artifact_paths["run_metadata"].write_text(
    json.dumps(run_metadata, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
_write_csv_exclusive(validation_df, artifact_paths["validation"])
print({
    "balanced_common_weight": selection_summary["common_knee_weight"],
    "validation_passed": validation_passed,
    "output_dir": str(OUTPUT_DIR.relative_to(REPO_ROOT)),
})


In [ ]:
failed_validation_checks = validation_df.loc[
    ~validation_df['passed'], 'check'
].tolist()
if failed_validation_checks:
    raise RuntimeError(
        f'Final sweep validation failed: {failed_validation_checks}'
    )

print({
    'balanced_common_weight': selection_summary['common_knee_weight'],
    'pv_metric_warning': selection_summary['pv_metric_warning'],
    'validation_passed': True,
    'output_dir': str(OUTPUT_DIR),
})
validation_df
